Simulamos nulos

In [3]:
# =============================================================================
# 01_simular_nulos_corregido_v2.py
#
# Paso 1: Simulación de Datos Faltantes
# CORRECCIÓN: Convierte las fechas de UTC a la hora local de cada ciudad.
# =============================================================================

import pandas as pd
import numpy as np
import os
import random
import time

# --- 0. Carga de Datos y Preparación Inicial ---
try:
    file_path_load = os.path.join('CSVs', 'Air_Quality.csv')
    if not os.path.exists(file_path_load):
        script_dir_parent = os.path.dirname(os.getcwd())
        alt_file_path_load = os.path.join(script_dir_parent, 'CSVs', 'Air_Quality.csv')
        if os.path.exists(alt_file_path_load):
            file_path_load = alt_file_path_load
        else:
            raise FileNotFoundError("Air_Quality.csv no encontrado.")
    df_air_quality_raw = pd.read_csv(file_path_load)
    print(f"Datos cargados desde: {file_path_load}")
except FileNotFoundError:
    print(f"Error: No se encontró el archivo Air_Quality.csv. Creando DataFrame de ejemplo...");
    # ... (código de ejemplo) ...
    # (El código de ejemplo no se modificará, ya que es solo un fallback)
    df_air_quality_raw = pd.DataFrame() # Simplificado para el ejemplo

if not df_air_quality_raw.empty:
    # Asegurar que la columna 'Date' es de tipo datetime y está localizada en UTC
    df_air_quality_raw['Date'] = pd.to_datetime(df_air_quality_raw['Date'], utc=True)

    if 'CO2' in df_air_quality_raw.columns:
        df_air_quality_raw = df_air_quality_raw.drop(columns=['CO2'])
        print("Columna 'CO2' descartada.")

    # --- 1. Conversión de Zona Horaria ---
    print("\nConvirtiendo fechas de UTC a la hora local de cada ciudad...")

    city_timezones = {
        'Brasilia': 'America/Sao_Paulo', # GMT-3
        'Cairo': 'Africa/Cairo',         # GMT+2
        'Dubai': 'Asia/Dubai',           # GMT+4
        'London': 'Europe/London',       # GMT+0/GMT+1 (Pandas maneja DST, pero aquí será consistente)
        'New York': 'America/New_York',    # GMT-5 (EST)
        'Sydney': 'Australia/Sydney'     # GMT+10 (AEST)
    }

    # Reemplazar "New_York" por "New York" si es necesario
    df_air_quality_raw['City'] = df_air_quality_raw['City'].replace('New_York', 'New York')

    df_parts = []
    for city, tz in city_timezones.items():
        df_city = df_air_quality_raw[df_air_quality_raw['City'] == city].copy()
        if not df_city.empty:
            df_city['Date'] = df_city['Date'].dt.tz_convert(tz)
            df_parts.append(df_city)
        else:
            print(f"  Advertencia: No se encontraron datos para la ciudad '{city}' en el mapeo de zonas horarias.")
    
    # Concatenar de nuevo y ordenar por ciudad y fecha
    df_air_quality = pd.concat(df_parts).sort_values(by=['City', 'Date']).reset_index(drop=True)

    print("Conversión de zona horaria completada.")
    # Verificación visual
    print("\nEjemplo de fechas convertidas (Brasilia):")
    print(df_air_quality[df_air_quality['City'] == 'Brasilia'].head(2)[['Date', 'City']])
    print("\nEjemplo de fechas convertidas (Sydney):")
    print(df_air_quality[df_air_quality['City'] == 'Sydney'].head(2)[['Date', 'City']])
else:
    # Si usamos el fallback de datos de ejemplo
    df_air_quality = df_air_quality_raw


# --- Parámetros de Simulación ---
CONTAMINANT_COLS = ['CO', 'NO2', 'SO2', 'O3', 'PM2.5', 'PM10']
AQI_COL = ['AQI']
ALL_TARGET_COLS = CONTAMINANT_COLS + AQI_COL
N_REPETITIONS = 5

NAN_SIMULATION_SCENARIOS = [
    {'name': 'RandomRows_20percent', 'type': 'row_percentage', 'config': {'percentage': 0.20}},
    {'name': 'BlockRows_12h_20percent', 'type': 'block_rows', 'config': {'length': 12, 'percentage': 0.20}},
    {'name': 'BlockRows_24h_20percent', 'type': 'block_rows', 'config': {'length': 24, 'percentage': 0.20}},
    {'name': 'BlockRows_36h_20percent', 'type': 'block_rows', 'config': {'length': 36, 'percentage': 0.20}},
    {'name': 'BlockRows_48h_20percent', 'type': 'block_rows', 'config': {'length': 48, 'percentage': 0.20}},
    {'name': 'BlockRows_60h_20percent', 'type': 'block_rows', 'config': {'length': 60, 'percentage': 0.20}}
]

PROJECT_ROOT_DIR = os.getcwd()
SIMULATED_DATA_ROOT_DIR = os.path.join(PROJECT_ROOT_DIR, 'CSVs', 'simulados') # Nueva carpeta de salida
os.makedirs(SIMULATED_DATA_ROOT_DIR, exist_ok=True)

# --- Funciones Auxiliares para Simulación de NaNs (sin cambios)---
def introduce_nans_by_row(df_original_city_input, scenario_config_input, random_seed_input):
    np.random.seed(random_seed_input); random.seed(random_seed_input)
    df_with_nans = df_original_city_input.copy().reset_index(drop=True)
    n_rows = len(df_with_nans); row_indices_to_nan_positions = []
    scenario_type, config = scenario_config_input['type'], scenario_config_input['config']
    if scenario_type == 'row_percentage':
        num_rows_to_nan = int(n_rows * config['percentage'])
        if num_rows_to_nan > 0 and n_rows > 0: row_indices_to_nan_positions = random.sample(range(n_rows), min(num_rows_to_nan, n_rows))
    elif scenario_type == 'block_rows':
        block_length, percentage = config['length'], config.get('percentage', 0.20); total_rows_to_nan = int(n_rows * percentage)
        num_blocks = max(1, total_rows_to_nan // block_length) if block_length > 0 else 0
        used_indices, possible_starts = set(), list(range(n_rows - block_length + 1)); random.shuffle(possible_starts)
        blocks_created = 0
        for start_pos in possible_starts:
            if blocks_created >= num_blocks: break
            block_range = set(range(start_pos, start_pos + block_length))
            if not block_range.intersection(used_indices):
                row_indices_to_nan_positions.extend(list(block_range)); used_indices.update(block_range); blocks_created += 1
        row_indices_to_nan_positions = sorted(list(set(row_indices_to_nan_positions)))
    if row_indices_to_nan_positions:
        col_positions_to_nan = df_with_nans.columns.get_indexer(ALL_TARGET_COLS)
        col_positions_to_nan = [pos for pos in col_positions_to_nan if pos != -1]
        if col_positions_to_nan: df_with_nans.iloc[row_indices_to_nan_positions, col_positions_to_nan] = np.nan
    return df_with_nans

# --- Script Principal de Simulación ---
start_time_script = time.time()

if 'df_air_quality' not in locals() or df_air_quality.empty:
    print("Error: df_air_quality no está cargado o está vacío. Saliendo."); exit()

for city_name_original in df_air_quality['City'].unique():
    city_name_safe = city_name_original.replace(" ", "_")
    print(f"\nProcesando Ciudad: {city_name_original} (Directorio seguro: {city_name_safe})")
    city_sim_data_dir_root = os.path.join(SIMULATED_DATA_ROOT_DIR, city_name_safe)
    os.makedirs(city_sim_data_dir_root, exist_ok=True)

    df_city_original_for_sim = df_air_quality[df_air_quality['City'] == city_name_original].copy()
    if df_city_original_for_sim.empty: print(f"  No hay datos para la ciudad {city_name_original}."); continue
    if df_city_original_for_sim[ALL_TARGET_COLS].isnull().any().any():
        print(f"  Advertencia: NaNs preexistentes en {city_name_original}. Rellenando con ffill/bfill.");
        for col_fill in ALL_TARGET_COLS:
             if col_fill in df_city_original_for_sim.columns: df_city_original_for_sim[col_fill] = df_city_original_for_sim[col_fill].ffill().bfill()
        if df_city_original_for_sim[ALL_TARGET_COLS].isnull().any().any():
            print(f"  Error: No se pudieron rellenar NaNs en {city_name_original}. Omitiendo."); continue

    for scenario in NAN_SIMULATION_SCENARIOS:
        scenario_name_safe = scenario['name'].replace("%", "percent")
        print(f"  Generando escenario: {scenario_name_safe}")
        scenario_city_sim_data_dir_specific = os.path.join(city_sim_data_dir_root, scenario_name_safe)
        os.makedirs(scenario_city_sim_data_dir_specific, exist_ok=True)
        for rep in range(1, N_REPETITIONS + 1):
            current_seed = int(time.time() * 1000 + random.randint(0,100000) + rep + hash(city_name_original) + hash(scenario_name_safe)) % (2**32 - 1)
            df_city_with_nans = introduce_nans_by_row(df_city_original_for_sim, scenario, random_seed_input=current_seed)
            sim_filename = f"{city_name_safe}_{scenario_name_safe}_rep{rep}_seed{current_seed}_with_nans.csv"
            df_city_with_nans.to_csv(os.path.join(scenario_city_sim_data_dir_specific, sim_filename), index=False)
    print(f"  Simulación de escenarios para {city_name_original} completada.")

end_time_script = time.time()
print(f"\nProceso de simulación de NaNs completado.")
print(f"Archivos de datos simulados guardados en subcarpetas dentro de: {SIMULATED_DATA_ROOT_DIR}")
print(f"Tiempo total de ejecución: {(end_time_script - start_time_script) / 60:.2f} minutos.")

Datos cargados desde: CSVs\Air_Quality.csv
Columna 'CO2' descartada.

Convirtiendo fechas de UTC a la hora local de cada ciudad...
Conversión de zona horaria completada.

Ejemplo de fechas convertidas (Brasilia):
                        Date      City
0  2023-12-31 21:00:00-03:00  Brasilia
1  2023-12-31 22:00:00-03:00  Brasilia

Ejemplo de fechas convertidas (Sydney):
                            Date    City
43920  2024-01-01 11:00:00+11:00  Sydney
43921  2024-01-01 12:00:00+11:00  Sydney

Procesando Ciudad: Brasilia (Directorio seguro: Brasilia)
  Generando escenario: RandomRows_20percent
  Generando escenario: BlockRows_12h_20percent
  Generando escenario: BlockRows_24h_20percent
  Generando escenario: BlockRows_36h_20percent
  Generando escenario: BlockRows_48h_20percent
  Generando escenario: BlockRows_60h_20percent
  Simulación de escenarios para Brasilia completada.

Procesando Ciudad: Cairo (Directorio seguro: Cairo)
  Generando escenario: RandomRows_20percent
  Generando escena

Interpolación Lineal

In [1]:
# =============================================================================
# 02_imputacion_lineal.py
#
# Paso 2: Imputación con Interpolación Lineal y Evaluación
#
# Esta celda lee los archivos con NaNs generados, les aplica interpolación 
# lineal y evalúa su rendimiento.
# =============================================================================

import pandas as pd
import numpy as np
import os
import random
from sklearn.metrics import mean_absolute_error, mean_squared_error
import time
import glob

# --- 0. Carga de Datos ORIGINALES (Ground Truth) ---
try:
    file_path_load = os.path.join('CSVs', 'Air_Quality.csv')
    if not os.path.exists(file_path_load):
        script_dir_parent = os.path.dirname(os.getcwd())
        alt_file_path_load = os.path.join(script_dir_parent, 'CSVs', 'Air_Quality.csv')
        if os.path.exists(alt_file_path_load):
            file_path_load = alt_file_path_load
        else:
            raise FileNotFoundError("Air_Quality.csv no encontrado.")
    df_air_quality_ground_truth = pd.read_csv(file_path_load)
    print(f"Datos 'Ground Truth' cargados desde: {file_path_load}")
except FileNotFoundError:
    print(f"Error: No se encontró el archivo Air_Quality.csv. Es necesario para la evaluación.")
    exit()

df_air_quality_ground_truth['Date'] = pd.to_datetime(df_air_quality_ground_truth['Date'])
if 'CO2' in df_air_quality_ground_truth.columns:
    df_air_quality_ground_truth = df_air_quality_ground_truth.drop(columns=['CO2'])

# --- Parámetros de Imputación y Evaluación ---
CONTAMINANT_COLS = ['CO', 'NO2', 'SO2', 'O3', 'PM2.5', 'PM10']
AQI_COL = ['AQI']
ALL_TARGET_COLS = CONTAMINANT_COLS + AQI_COL

# --- Lista de escenarios a procesar (DEBE COINCIDIR CON LA CELDA 1) ---
NAN_SIMULATION_SCENARIOS = [
    {'name': 'RandomRows_20percent', 'type': 'row_percentage', 'config': {'percentage': 0.20}},
    {'name': 'BlockRows_12h_20percent', 'type': 'block_rows', 'config': {'length': 12, 'percentage': 0.20}},
    {'name': 'BlockRows_24h_20percent', 'type': 'block_rows', 'config': {'length': 24, 'percentage': 0.20}},
    {'name': 'BlockRows_36h_20percent', 'type': 'block_rows', 'config': {'length': 36, 'percentage': 0.20}},
    {'name': 'BlockRows_48h_20percent', 'type': 'block_rows', 'config': {'length': 48, 'percentage': 0.20}},
    {'name': 'BlockRows_60h_20percent', 'type': 'block_rows', 'config': {'length': 60, 'percentage': 0.20}}
]

PROJECT_ROOT_DIR = os.getcwd()
SIMULATED_DATA_ROOT_DIR = os.path.join(PROJECT_ROOT_DIR, 'CSVs', 'simulados')
IMPUTATIONS_BASE_DIR = os.path.join(PROJECT_ROOT_DIR, 'imputaciones')
INTERPOLATION_METHOD_NAME = 'interpolacionLineal'
METRICS_OUTPUT_DIR = os.path.join(IMPUTATIONS_BASE_DIR, INTERPOLATION_METHOD_NAME, 'metricas')
IMPUTED_CSVS_OUTPUT_DIR = os.path.join(IMPUTATIONS_BASE_DIR, INTERPOLATION_METHOD_NAME, 'CSVs')

os.makedirs(METRICS_OUTPUT_DIR, exist_ok=True)
os.makedirs(IMPUTED_CSVS_OUTPUT_DIR, exist_ok=True)

# --- Funciones Auxiliares para Métricas ---
def calculate_metrics_with_nmae_nrmse(y_true_df_reset, y_pred_df_reset, nan_mask_for_metrics_df_reset):
    results = {}
    for col in ALL_TARGET_COLS:
        results[col] = {'MAE': np.nan, 'RMSE': np.nan, 'NRMAE_range': np.nan, 'NRMSE_range': np.nan}
        if col not in y_true_df_reset.columns or col not in y_pred_df_reset.columns or col not in nan_mask_for_metrics_df_reset.columns:
            continue

        true_col_vals = y_true_df_reset[col].values
        pred_col_vals = y_pred_df_reset[col].values
        mask_col_flags = nan_mask_for_metrics_df_reset[col].values

        true_at_nans = true_col_vals[mask_col_flags]
        pred_at_nans = pred_col_vals[mask_col_flags]

        if len(true_at_nans) == 0: continue
        valid_pred_mask = ~np.isnan(pred_at_nans)
        true_at_nans_valid = true_at_nans[valid_pred_mask]
        pred_at_nans_valid = pred_at_nans[valid_pred_mask]

        if len(true_at_nans_valid) == 0: continue
        try:
            mae = mean_absolute_error(true_at_nans_valid, pred_at_nans_valid)
            rmse = np.sqrt(mean_squared_error(true_at_nans_valid, pred_at_nans_valid))
            original_col_range = y_true_df_reset[col].max() - y_true_df_reset[col].min()
            
            nrmae_range = np.nan
            nrmse_range = np.nan
            if original_col_range > 1e-9:
                nrmae_range = mae / original_col_range
                nrmse_range = rmse / original_col_range

            results[col]['MAE'] = mae; results[col]['RMSE'] = rmse
            results[col]['NRMAE_range'] = nrmae_range; results[col]['NRMSE_range'] = nrmse_range
        except Exception as e: pass
    return results

# --- Script Principal de Imputación y Evaluación ---
all_detailed_metrics_results = []
start_time_script = time.time()

for city_name_original in df_air_quality_ground_truth['City'].unique():
    city_name_safe = city_name_original.replace(" ", "_").replace("-", "_")
    print(f"\nProcesando Ciudad: {city_name_original}")
    
    df_city_gt = df_air_quality_ground_truth[df_air_quality_ground_truth['City'] == city_name_original].copy()
    if df_city_gt[ALL_TARGET_COLS].isnull().any().any():
        for col_fill in ALL_TARGET_COLS:
            if col_fill in df_city_gt.columns: df_city_gt[col_fill] = df_city_gt[col_fill].ffill().bfill()
    if df_city_gt.isnull().values.any():
        print(f"  No se pudieron rellenar NaNs en el ground truth de {city_name_original}. Omitiendo ciudad.")
        continue
    df_city_original_ground_truth_reset = df_city_gt.reset_index(drop=True)

    sim_data_city_dir = os.path.join(SIMULATED_DATA_ROOT_DIR, city_name_safe)
    if not os.path.exists(sim_data_city_dir):
        print(f"  Directorio de datos simulados no encontrado para {city_name_original}. Omitiendo.")
        continue

    for scenario in NAN_SIMULATION_SCENARIOS:
        scenario_name_safe = scenario['name'].replace("%", "percent").replace(" ", "_")
        print(f"  Evaluando escenario: {scenario_name_safe}")
        
        sim_data_scenario_dir = os.path.join(sim_data_city_dir, scenario_name_safe)
        if not os.path.exists(sim_data_scenario_dir):
            print(f"    Directorio de escenario no encontrado: {sim_data_scenario_dir}. Omitiendo.")
            continue
            
        imputed_csv_scenario_dir = os.path.join(IMPUTED_CSVS_OUTPUT_DIR, city_name_safe, scenario_name_safe)
        os.makedirs(imputed_csv_scenario_dir, exist_ok=True)

        repetition_files = glob.glob(os.path.join(sim_data_scenario_dir, "*_with_nans.csv"))

        for sim_file_path in repetition_files:
            file_name = os.path.basename(sim_file_path)
            try:
                df_city_with_nans_reset = pd.read_csv(sim_file_path)
                rep_str = file_name.split('_rep')[-1].split('_')[0]
                seed_str = file_name.split('_seed')[-1].split('_')[0]
                rep = int(rep_str) if rep_str.isdigit() else 0
                current_seed = int(seed_str) if seed_str.isdigit() else 0
            except Exception as e:
                print(f"      Error cargando {file_name}: {e}")
                continue
            
            # 1. Realizar imputación
            df_city_imputed_linear_reset = df_city_with_nans_reset.copy()
            for col in ALL_TARGET_COLS:
                if col in df_city_imputed_linear_reset.columns:
                    df_city_imputed_linear_reset[col] = df_city_imputed_linear_reset[col].interpolate(
                        method='linear', limit_direction='both', limit_area=None)
            for col in ALL_TARGET_COLS:
                if col in df_city_imputed_linear_reset.columns:
                    df_city_imputed_linear_reset[col] = df_city_imputed_linear_reset[col].ffill().bfill()

            # 2. Guardar archivo imputado
            imputed_filename = file_name.replace('_with_nans.csv', f'_imputed_{INTERPOLATION_METHOD_NAME}.csv')
            df_city_imputed_linear_reset.to_csv(os.path.join(imputed_csv_scenario_dir, imputed_filename), index=False)

            # 3. Calcular métricas
            nan_mask = df_city_with_nans_reset[ALL_TARGET_COLS].isnull() & ~df_city_original_ground_truth_reset[ALL_TARGET_COLS].isnull()
            metrics_scores = calculate_metrics_with_nmae_nrmse(df_city_original_ground_truth_reset, df_city_imputed_linear_reset, nan_mask)

            # 4. Almacenar resultados detallados
            for var_name in ALL_TARGET_COLS:
                all_detailed_metrics_results.append({
                    'City': city_name_original, 'Scenario': scenario_name_safe, 'Repetition': rep,
                    'Seed': current_seed, 'Imputation_Method': INTERPOLATION_METHOD_NAME,
                    'Variable': var_name, 
                    'MAE': metrics_scores[var_name]['MAE'], 
                    'RMSE': metrics_scores[var_name]['RMSE'],
                    'NRMAE_range': metrics_scores[var_name]['NRMAE_range'],
                    'NRMSE_range': metrics_scores[var_name]['NRMSE_range']
                })
    print(f"  Procesamiento de escenarios para {city_name_original} completado.")

# --- Guardado de Métricas Resumidas ---
if all_detailed_metrics_results:
    df_all_detailed_metrics = pd.DataFrame(all_detailed_metrics_results)
    
    print("\nGenerando resúmenes de métricas por variable y ciudad...")
    grouped_by_city_var = df_all_detailed_metrics.groupby(['City', 'Variable'])[['MAE', 'RMSE', 'NRMAE_range', 'NRMSE_range']].mean().reset_index()
    grouped_by_city_var.rename(columns={'MAE': 'Avg_MAE', 'RMSE': 'Avg_RMSE', 'NRMAE_range': 'Avg_NRMAE_range', 'NRMSE_range': 'Avg_NRMSE_range'}, inplace=True)
    for city_name_original in grouped_by_city_var['City'].unique():
        city_name_safe = city_name_original.replace(" ", "_").replace("-", "_")
        city_metrics_df = grouped_by_city_var[grouped_by_city_var['City'] == city_name_original]
        variables_summary_path = os.path.join(METRICS_OUTPUT_DIR, f'metrics_variables_summary_{city_name_safe}.csv')
        city_metrics_df.to_csv(variables_summary_path, index=False)
        print(f"  Resumen de variables para {city_name_original} guardado en: {variables_summary_path}")
        if city_name_original == df_air_quality_ground_truth['City'].unique()[0]: print(city_metrics_df)

    print("\nGenerando resumen combinado de contaminantes...")
    df_contaminants_only = df_all_detailed_metrics[df_all_detailed_metrics['Variable'].isin(CONTAMINANT_COLS)]
    contaminants_combined_summary = df_contaminants_only.groupby('City')[['MAE', 'RMSE', 'NRMAE_range', 'NRMSE_range']].mean().reset_index()
    contaminants_combined_summary.rename(columns={'MAE': 'Avg_MAE_Conts', 'RMSE': 'Avg_RMSE_Conts', 'NRMAE_range': 'Avg_NRMAE_range_Conts', 'NRMSE_range': 'Avg_NRMSE_range_Conts'}, inplace=True)
    contaminants_summary_path = os.path.join(METRICS_OUTPUT_DIR, 'metrics_contaminants_combined_summary.csv')
    contaminants_combined_summary.to_csv(contaminants_summary_path, index=False)
    print(f"  Resumen combinado de contaminantes guardado en: {contaminants_summary_path}")
    print(contaminants_combined_summary)

    print("\nGenerando resumen combinado de AQI...")
    df_aqi_only = df_all_detailed_metrics[df_all_detailed_metrics['Variable'] == 'AQI']
    aqi_combined_summary = df_aqi_only.groupby('City')[['MAE', 'RMSE', 'NRMAE_range', 'NRMSE_range']].mean().reset_index()
    aqi_combined_summary.rename(columns={'MAE': 'Avg_MAE_AQI', 'RMSE': 'Avg_RMSE_AQI', 'NRMAE_range': 'Avg_NRMAE_range_AQI', 'NRMSE_range': 'Avg_NRMSE_range_AQI'}, inplace=True)
    aqi_summary_path = os.path.join(METRICS_OUTPUT_DIR, 'metrics_aqi_combined_summary.csv')
    aqi_combined_summary.to_csv(aqi_summary_path, index=False)
    print(f"  Resumen combinado de AQI guardado en: {aqi_summary_path}")
    print(aqi_combined_summary)
else:
    print("No se generaron resultados de métricas detalladas.")

end_time_script = time.time()
print(f"\nProceso de imputación y evaluación completado.")
print(f"Archivos de datos simulados leídos desde: {SIMULATED_DATA_ROOT_DIR}")
print(f"Archivos CSV imputados guardados en subcarpetas dentro de: {IMPUTED_CSVS_OUTPUT_DIR}")
print(f"Archivos de métricas resumidas guardados en: {METRICS_OUTPUT_DIR}")
print(f"Tiempo total de ejecución: {(end_time_script - start_time_script) / 60:.2f} minutos.")

Datos 'Ground Truth' cargados desde: CSVs\Air_Quality.csv

Procesando Ciudad: Brasilia
  Evaluando escenario: RandomRows_20percent
  Evaluando escenario: BlockRows_12h_20percent
  Evaluando escenario: BlockRows_24h_20percent
  Evaluando escenario: BlockRows_36h_20percent
  Evaluando escenario: BlockRows_48h_20percent
  Evaluando escenario: BlockRows_60h_20percent
  Procesamiento de escenarios para Brasilia completado.

Procesando Ciudad: Cairo
  Evaluando escenario: RandomRows_20percent
  Evaluando escenario: BlockRows_12h_20percent
  Evaluando escenario: BlockRows_24h_20percent
  Evaluando escenario: BlockRows_36h_20percent
  Evaluando escenario: BlockRows_48h_20percent
  Evaluando escenario: BlockRows_60h_20percent
  Procesamiento de escenarios para Cairo completado.

Procesando Ciudad: Dubai
  Evaluando escenario: RandomRows_20percent
  Evaluando escenario: BlockRows_12h_20percent
  Evaluando escenario: BlockRows_24h_20percent
  Evaluando escenario: BlockRows_36h_20percent
  Evaluan

SARIMA ORDERS

In [ ]:
# CELDA 1: BÚSQUEDA DE ÓRDENES ÓPTIMOS ARIMA/SARIMA (EJECUTAR SOLO UNA VEZ O CUANDO SEA NECESARIO)
import pandas as pd
import numpy as np
import os
import pmdarima as pm
import json
import warnings
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

# --- 0. Carga de Datos (Asegúrate de que df_air_quality esté cargado) ---
try:
    file_path_load = os.path.join('CSVs', 'Air_Quality.csv')
    if not os.path.exists(file_path_load):
        script_dir_parent = os.path.dirname(os.getcwd())
        alt_file_path_load = os.path.join(script_dir_parent, 'CSVs', 'Air_Quality.csv')
        if os.path.exists(alt_file_path_load): file_path_load = alt_file_path_load
        else: raise FileNotFoundError("Air_Quality.csv no encontrado.")
    df_air_quality = pd.read_csv(file_path_load)
    df_air_quality['Date'] = pd.to_datetime(df_air_quality['Date'])
    if 'CO2' in df_air_quality.columns: df_air_quality = df_air_quality.drop(columns=['CO2'])
    print(f"Datos cargados desde: {file_path_load} para Celda 1")
except FileNotFoundError:
    print("Error: df_air_quality no encontrado en Celda 1.")
    _example_data = [] # Datos de ejemplo mínimos
    _cities_example = ['Brasilia', 'Cairo']
    _start_date = pd.to_datetime('2024-01-01')
    for _city_ex in _cities_example:
        for i in range(24*7*4): # 4 semanas de datos horarios
            _date = _start_date + pd.Timedelta(hours=i)
            _example_data.append([_date, _city_ex, 200+np.random.rand()*100, 20+np.random.rand()*10, 40+np.random.rand()*20 ])
    df_air_quality = pd.DataFrame(_example_data, columns=['Date', 'City', 'CO', 'NO2', 'AQI'])
    print("DataFrame de ejemplo creado para Celda 1.")


# --- Parámetros ---
COLUMNS_FOR_ARIMA = ['CO', 'NO2', 'SO2', 'O3', 'PM2.5', 'PM10', 'AQI']
SEASONAL_PERIOD = 24

PROJECT_ROOT_DIR_ARIMA_SEARCH = os.getcwd()
# Directorio donde se guardarán los JSONs directamente
ORDERS_OUTPUT_DIR = os.path.join(PROJECT_ROOT_DIR_ARIMA_SEARCH, 'imputaciones', 'SARIMA_auto', 'model_orders_v2')
os.makedirs(ORDERS_OUTPUT_DIR, exist_ok=True) # Solo se crea la carpeta base 'model_orders'

print(f"--- INICIO DE BÚSQUEDA DE ÓRDENES ARIMA/SARIMA ---")
print(f"Los órdenes se guardarán directamente en: {ORDERS_OUTPUT_DIR}")

if 'df_air_quality' not in locals() or df_air_quality.empty:
    print("Error: df_air_quality no está definido o está vacío en Celda 1.")
    exit()

for city_name_original in df_air_quality['City'].unique():
    city_name_safe = city_name_original.replace(" ", "_").replace("-", "_")
    print(f"\nProcesando ciudad: {city_name_original}")

    df_city = df_air_quality[df_air_quality['City'] == city_name_original].copy()
    if df_city.empty:
        print(f"  No hay datos para {city_name_original}, omitiendo.")
        continue

    for col_name in COLUMNS_FOR_ARIMA:
        if col_name not in df_city.columns:
            print(f"  Variable {col_name} no encontrada en los datos de {city_name_original}, omitiendo.")
            continue

        print(f"  Buscando orden para variable: {col_name}...")
        series_data = df_city[col_name].dropna()

        if len(series_data) < 2 * SEASONAL_PERIOD:
            print(f"    Datos insuficientes para SARIMA ({len(series_data)} puntos). Intentando ARIMA no estacional.")
            try:
                auto_model = pm.auto_arima(series_data, start_p=1, start_q=1, max_p=5, max_q=5, d=None, seasonal=False, trace=False, error_action='ignore', suppress_warnings=True, stepwise=True)
                order = auto_model.order; seasonal_order = (0,0,0,0)
            except Exception as e:
                print(f"    Error en auto_arima no estacional para {col_name}: {e}. Usando (1,0,0).")
                order = (1,0,0); seasonal_order = (0,0,0,0)
        elif len(series_data) < 10:
             print(f"    Datos muy insuficientes (<10 puntos). Usando (1,0,0).")
             order = (1,0,0); seasonal_order = (0,0,0,0)
        else:
            try:
                auto_model = pm.auto_arima(series_data, start_p=1, start_q=1, max_p=5, max_q=5, m=SEASONAL_PERIOD, start_P=0, D=None, max_P=2, max_Q=2, seasonal=True, trace=False, error_action='ignore', suppress_warnings=True, stepwise=True)     
                order = auto_model.order; seasonal_order = auto_model.seasonal_order
                print(f"    Orden encontrado para {col_name}: {order}, Estacional: {seasonal_order}")
            except Exception as e:
                print(f"    Error en auto_arima para {col_name}: {e}. Usando (1,0,1)(1,0,1,{SEASONAL_PERIOD}).")
                order = (1,0,1); seasonal_order = (1,0,1,SEASONAL_PERIOD) if SEASONAL_PERIOD > 0 else (0,0,0,0)
        
        order_data = {'order': list(order), 'seasonal_order': list(seasonal_order)}
        # Guardar directamente en ORDERS_OUTPUT_DIR, sin subcarpeta de ciudad para el JSON
        file_path_order = os.path.join(ORDERS_OUTPUT_DIR, f'orders_{city_name_safe}_{col_name}.json')
        with open(file_path_order, 'w') as f:
            json.dump(order_data, f, indent=4)
        print(f"    Órdenes guardados en: {file_path_order}")

print(f"--- FIN DE BÚSQUEDA DE ÓRDENES ARIMA/SARIMA ---")

Datos originales (ground truth) cargados desde: CSVs\Air_Quality.csv
--- Fase 1: Encontrando/Cargando Órdenes SARIMA Óptimos ---
  Procesando ciudad para encontrar órdenes: Brasilia
    Procesando variable para auto_arima: CO
      Ejecutando auto_arima para Brasilia-CO (esto puede tardar)...
        Usando subconjunto de 2160 filas para auto_arima.
      auto_arima completado para Brasilia-CO: Order=(3, 0, 0), Seasonal=(1, 0, 1, 24)
    Procesando variable para auto_arima: NO2
      Ejecutando auto_arima para Brasilia-NO2 (esto puede tardar)...
        Usando subconjunto de 2160 filas para auto_arima.
      auto_arima completado para Brasilia-NO2: Order=(1, 1, 0), Seasonal=(1, 0, 0, 24)
    Procesando variable para auto_arima: SO2
      Ejecutando auto_arima para Brasilia-SO2 (esto puede tardar)...
        Usando subconjunto de 2160 filas para auto_arima.
      auto_arima completado para Brasilia-SO2: Order=(3, 1, 1), Seasonal=(1, 0, 0, 24)
    Procesando variable para auto_arima: O3


KeyboardInterrupt: 

Imputación SARIMA

In [1]:
# =============================================================================
# 02_imputacion_sarima.py (Versión con Target City Opcional)
# =============================================================================

import pandas as pd
import numpy as np
import os
import json
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error, mean_squared_error
import time
import warnings
import glob

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# --- Parámetros de Imputación y Evaluación ---
# CORRECCIÓN: Añadir parámetro de ciudad objetivo
TARGET_CITY_SARIMA = None  # Poner a None para ejecutar para TODAS las ciudades

COLUMNS_TO_IMPUTE_SARIMA = ['CO', 'NO2', 'SO2', 'O3', 'PM2.5', 'PM10', 'AQI']
N_REPETITIONS_SARIMA = 5

NAN_SIMULATION_SCENARIOS_SARIMA = [
    {'name': 'RandomRows_20percent'},
    {'name': 'BlockRows_12h_20percent'},
    {'name': 'BlockRows_24h_20percent'},
    {'name': 'BlockRows_36h_20percent'},
    {'name': 'BlockRows_48h_20percent'},
    {'name': 'BlockRows_60h_20percent'}
]

# --- Rutas de Entrada y Salida ---
PROJECT_ROOT_DIR_SARIMA_IMPUTE = os.getcwd()
SIMULATED_DATA_INPUT_ROOT_DIR = os.path.join(PROJECT_ROOT_DIR_SARIMA_IMPUTE, 'CSVs', 'simulados')
ORDERS_JSON_INPUT_DIR = os.path.join(PROJECT_ROOT_DIR_SARIMA_IMPUTE, 'imputaciones', 'SARIMA', 'model_orders')
SARIMA_METHOD_NAME = 'SARIMA_full'
IMPUTATIONS_SARIMA_BASE_DIR = os.path.join(PROJECT_ROOT_DIR_SARIMA_IMPUTE, 'imputaciones', SARIMA_METHOD_NAME)
METRICS_SARIMA_OUTPUT_DIR = os.path.join(IMPUTATIONS_SARIMA_BASE_DIR, 'metricas')
IMPUTED_CSVS_SARIMA_OUTPUT_DIR = os.path.join(IMPUTATIONS_SARIMA_BASE_DIR, 'CSVs')
os.makedirs(METRICS_SARIMA_OUTPUT_DIR, exist_ok=True)
os.makedirs(IMPUTED_CSVS_SARIMA_OUTPUT_DIR, exist_ok=True)

# --- Función de Imputación SARIMA (sin cambios) ---
def impute_sarima_kalman_smoother(series_with_nans, order_tuple, seasonal_order_tuple, original_series_for_bounds=None):
    imputed_series = series_with_nans.copy()
    original_nan_mask = imputed_series.isnull()
    if not original_nan_mask.any(): return series_with_nans
    min_len_heuristic = max(sum(order_tuple), sum(seasonal_order_tuple[:3])) + seasonal_order_tuple[3] if seasonal_order_tuple[3] > 0 else max(sum(order_tuple), sum(seasonal_order_tuple[:3])); min_len_heuristic = max(min_len_heuristic, 10)
    if len(imputed_series.dropna()) < min_len_heuristic + 5: return series_with_nans.interpolate(method='linear', limit_direction='both').ffill().bfill()
    sarima_failed_or_unstable = False
    try:
        model = SARIMAX(imputed_series, order=order_tuple, seasonal_order=seasonal_order_tuple, enforce_stationarity=False, enforce_invertibility=False, initialization='approximate_diffuse')
        results = model.fit(disp=False, maxiter=200)
        predictions_all = results.predict(start=imputed_series.index[0], end=imputed_series.index[-1])
        imputed_series.loc[original_nan_mask] = predictions_all[original_nan_mask]
        if imputed_series.isnull().any():
            temp_linear_fill = series_with_nans.interpolate(method='linear', limit_direction='both'); imputed_series = imputed_series.fillna(temp_linear_fill).ffill().bfill()
    except Exception as e: sarima_failed_or_unstable = True
    if sarima_failed_or_unstable: imputed_series = series_with_nans.interpolate(method='linear', limit_direction='both').ffill().bfill()
    if original_series_for_bounds is not None and original_nan_mask.any():
        clean_original_data = original_series_for_bounds.dropna()
        if not clean_original_data.empty:
            q01, q99 = clean_original_data.quantile(0.001), clean_original_data.quantile(0.999); iqr_val = clean_original_data.quantile(0.75) - clean_original_data.quantile(0.25)
            padding = max(1.0, 0.1 * abs(q99 - q01)) if (q99 - q01) != 0 else 1.0
            lower_bound, upper_bound = q01 - 3 * iqr_val - padding, q99 + 3 * iqr_val + padding
            extreme_values_mask = original_nan_mask & ((imputed_series < lower_bound) | (imputed_series > upper_bound))
            if extreme_values_mask.any():
                temp_linear_imputation = series_with_nans.interpolate(method='linear', limit_direction='both').ffill().bfill(); imputed_series.loc[extreme_values_mask] = temp_linear_imputation[extreme_values_mask]
    return imputed_series

# --- Función para calcular métricas (sin cambios) ---
def calculate_metrics_sarima_full(y_true_df_reset, y_pred_df_reset, nan_mask_for_metrics_df_reset):
    results = {}
    for col in COLUMNS_TO_IMPUTE_SARIMA:
        results[col] = {'MAE': np.nan, 'RMSE': np.nan, 'NRMAE_range': np.nan, 'NRMSE_range': np.nan}
        if col not in y_true_df_reset.columns or col not in y_pred_df_reset.columns: continue
        true_col_vals, pred_col_vals, mask_col_flags = y_true_df_reset[col].values, y_pred_df_reset[col].values, nan_mask_for_metrics_df_reset[col].values
        true_at_nans, pred_at_nans = true_col_vals[mask_col_flags], pred_col_vals[mask_col_flags]
        if len(true_at_nans) == 0: continue
        valid_pred_mask = ~np.isnan(pred_at_nans); true_at_nans_valid, pred_at_nans_valid = true_at_nans[valid_pred_mask], pred_at_nans[valid_pred_mask]
        if len(true_at_nans_valid) == 0: continue
        try:
            mae, rmse = mean_absolute_error(true_at_nans_valid, pred_at_nans_valid), np.sqrt(mean_squared_error(true_at_nans_valid, pred_at_nans_valid))
            original_col_range = y_true_df_reset[col].max() - y_true_df_reset[col].min()
            nrmae_range = np.nan if original_col_range < 1e-9 else mae / original_col_range; nrmse_range = np.nan if original_col_range < 1e-9 else rmse / original_col_range
            results[col].update({'MAE': mae, 'RMSE': rmse, 'NRMAE_range': nrmae_range, 'NRMSE_range': nrmse_range})
        except Exception: pass
    return results

# --- Script Principal de Imputación SARIMA ---
all_detailed_metrics_sarima_results = []
start_time_script_sarima = time.time()
try:
    file_path_load_orig = os.path.join('CSVs', 'Air_Quality.csv')
    if not os.path.exists(file_path_load_orig):
        script_dir_parent = os.path.dirname(os.getcwd()); alt_file_path_load_orig = os.path.join(script_dir_parent, 'CSVs', 'Air_Quality.csv')
        file_path_load_orig = alt_file_path_load_orig if os.path.exists(alt_file_path_load_orig) else ''
    if not file_path_load_orig: raise FileNotFoundError("Air_Quality.csv (original) no encontrado.")
    df_air_quality_original_gt = pd.read_csv(file_path_load_orig); df_air_quality_original_gt['Date'] = pd.to_datetime(df_air_quality_original_gt['Date'])
    if 'CO2' in df_air_quality_original_gt.columns: df_air_quality_original_gt = df_air_quality_original_gt.drop(columns=['CO2'])
    print(f"Datos originales GT cargados desde: {file_path_load_orig}")
except FileNotFoundError:
    print("Error crítico: Archivo 'Air_Quality.csv' de ground truth no encontrado."); exit()

# CORRECCIÓN: Lógica para seleccionar la(s) ciudad(es) a procesar
if TARGET_CITY_SARIMA:
    cities_to_process = [TARGET_CITY_SARIMA]
    if TARGET_CITY_SARIMA not in df_air_quality_original_gt['City'].unique():
        print(f"Advertencia: La ciudad objetivo '{TARGET_CITY_SARIMA}' no se encuentra en el dataset. No se procesará nada.")
        cities_to_process = []
else:
    cities_to_process = df_air_quality_original_gt['City'].unique()

for city_name_original_sarima in cities_to_process:
    city_name_safe_sarima = city_name_original_sarima.replace(" ", "_").replace("-", "_")
    print(f"\nProcesando Ciudad para Imputación SARIMA: {city_name_original_sarima}")

    city_sim_data_input_dir_root = os.path.join(SIMULATED_DATA_INPUT_ROOT_DIR, city_name_safe_sarima)
    city_imputed_csv_sarima_dir_root = os.path.join(IMPUTED_CSVS_SARIMA_OUTPUT_DIR, city_name_safe_sarima)
    os.makedirs(city_imputed_csv_sarima_dir_root, exist_ok=True)

    df_city_gt_for_metrics = df_air_quality_original_gt[df_air_quality_original_gt['City'] == city_name_original_sarima].copy()
    if df_city_gt_for_metrics.empty: print(f"No hay datos para {city_name_original_sarima}. Omitiendo."); continue
    if df_city_gt_for_metrics[COLUMNS_TO_IMPUTE_SARIMA].isnull().any().any():
        for col_fill_gt in COLUMNS_TO_IMPUTE_SARIMA:
            if col_fill_gt in df_city_gt_for_metrics.columns: df_city_gt_for_metrics[col_fill_gt] = df_city_gt_for_metrics[col_fill_gt].ffill().bfill()
    df_city_gt_for_metrics_reset = df_city_gt_for_metrics.reset_index(drop=True)

    for scenario_sarima in NAN_SIMULATION_SCENARIOS_SARIMA:
        scenario_name_safe_sarima = scenario_sarima['name'].replace("%", "percent").replace(" ", "_")
        print(f"  Escenario: {scenario_name_safe_sarima}")
        scenario_city_sim_data_input_dir = os.path.join(city_sim_data_input_dir_root, scenario_name_safe_sarima)
        scenario_city_imputed_csv_sarima_dir = os.path.join(city_imputed_csv_sarima_dir_root, scenario_name_safe_sarima); os.makedirs(scenario_city_imputed_csv_sarima_dir, exist_ok=True)

        for rep_s in range(1, N_REPETITIONS_SARIMA + 1):
            sim_files_pattern = os.path.join(scenario_city_sim_data_input_dir, f"{city_name_safe_sarima}_{scenario_name_safe_sarima}_rep{rep_s}_*_with_nans.csv")
            sim_files_in_dir = sorted(glob.glob(sim_files_pattern))
            if not sim_files_in_dir: continue
            
            sim_file_path_to_load = sim_files_in_dir[0]; current_seed_loaded = os.path.basename(sim_file_path_to_load).split('_seed')[-1].split('_with_nans.csv')[0]
            try: df_city_with_nans_loaded = pd.read_csv(sim_file_path_to_load)
            except FileNotFoundError: print(f"    Error: Archivo simulado no encontrado: {sim_file_path_to_load}"); continue
            
            df_city_imputed_sarima = df_city_with_nans_loaded.copy()
            mask_for_metrics_s = pd.DataFrame(False, index=df_city_gt_for_metrics_reset.index, columns=COLUMNS_TO_IMPUTE_SARIMA)
            for col_mask in COLUMNS_TO_IMPUTE_SARIMA:
                 if col_mask in df_city_with_nans_loaded.columns and col_mask in df_city_gt_for_metrics_reset.columns:
                    mask_for_metrics_s[col_mask] = df_city_with_nans_loaded[col_mask].isnull() & ~df_city_gt_for_metrics_reset[col_mask].isnull()

            for col_impute_s in COLUMNS_TO_IMPUTE_SARIMA:
                print(f"    Imputando columna: {col_impute_s}...")
                order_file_path = os.path.join(ORDERS_JSON_INPUT_DIR, f'orders_{city_name_safe_sarima}_{col_impute_s}.json')
                try:
                    with open(order_file_path, 'r') as f: orders = json.load(f)
                    order_tuple, seasonal_order_tuple = tuple(orders['order']), tuple(orders['seasonal_order'])
                except (FileNotFoundError, json.JSONDecodeError):
                    print(f"      Archivo de órdenes no encontrado/corrupto para {col_impute_s}. Usando (1,0,1)(0,0,0,0)."); order_tuple, seasonal_order_tuple = (1,0,1), (0,0,0,0)
                series_to_impute, original_clean_series_for_bounds = df_city_with_nans_loaded[col_impute_s].copy(), df_city_gt_for_metrics_reset[col_impute_s].copy()
                imputed_values_col = impute_sarima_kalman_smoother(series_to_impute, order_tuple, seasonal_order_tuple, original_clean_series_for_bounds)
                df_city_imputed_sarima[col_impute_s] = imputed_values_col
            
            imputed_sarima_filename = f"{city_name_safe_sarima}_{scenario_name_safe_sarima}_rep{rep_s}_seed{current_seed_loaded}_imputed_sarima.csv"
            df_city_imputed_sarima.to_csv(os.path.join(scenario_city_imputed_csv_sarima_dir, imputed_sarima_filename), index=False)
            metrics_scores = calculate_metrics_sarima_full(df_city_gt_for_metrics_reset, df_city_imputed_sarima, mask_for_metrics_s)
            
            for var_name in COLUMNS_TO_IMPUTE_SARIMA:
                all_detailed_metrics_sarima_results.append({'City': city_name_original_sarima, 'Scenario': scenario_name_safe_sarima, 'Repetition': rep_s, 'Seed': current_seed_loaded, 'Imputation_Method': SARIMA_METHOD_NAME, 'Variable': var_name, 'MAE': metrics_scores[var_name]['MAE'], 'RMSE': metrics_scores[var_name]['RMSE'], 'NRMAE_range': metrics_scores[var_name]['NRMAE_range'], 'NRMSE_range': metrics_scores[var_name]['NRMSE_range']})

# --- Guardado de Métricas Resumidas ---
if all_detailed_metrics_sarima_results:
    df_all_detailed_metrics = pd.DataFrame(all_detailed_metrics_sarima_results)
    
    print("\nGenerando resúmenes de métricas por variable y ciudad...")
    grouped_by_city_var = df_all_detailed_metrics.groupby(['City', 'Variable'])[['MAE', 'RMSE', 'NRMAE_range', 'NRMSE_range']].mean().reset_index()
    grouped_by_city_var.rename(columns={'MAE': 'Avg_MAE', 'RMSE': 'Avg_RMSE', 'NRMAE_range': 'Avg_NRMAE_range', 'NRMSE_range': 'Avg_NRMSE_range'}, inplace=True)
    for city_name in grouped_by_city_var['City'].unique():
        city_name_safe = city_name.replace(" ", "_").replace("-", "_")
        city_metrics_df = grouped_by_city_var[grouped_by_city_var['City'] == city_name]
        variables_summary_path = os.path.join(METRICS_SARIMA_OUTPUT_DIR, f'metrics_variables_summary_{city_name_safe}.csv')
        city_metrics_df.to_csv(variables_summary_path, index=False); print(f"  Resumen de variables para {city_name} guardado en: {variables_summary_path}")

    print("\nGenerando resumen combinado de contaminantes...")
    # CORRECCIÓN: Filtrar correctamente las columnas de contaminantes
    contaminant_cols = [c for c in COLUMNS_TO_IMPUTE_SARIMA if c != 'AQI']
    df_contaminants_only = df_all_detailed_metrics[df_all_detailed_metrics['Variable'].isin(contaminant_cols)]
    contaminants_combined_summary = df_contaminants_only.groupby('City')[['MAE', 'RMSE', 'NRMAE_range', 'NRMSE_range']].mean().reset_index()
    contaminants_combined_summary.rename(columns={'MAE': 'Avg_MAE_Conts', 'RMSE': 'Avg_RMSE_Conts', 'NRMAE_range': 'Avg_NRMAE_range_Conts', 'NRMSE_range': 'Avg_NRMSE_range_Conts'}, inplace=True)
    contaminants_summary_path = os.path.join(METRICS_SARIMA_OUTPUT_DIR, 'metrics_contaminants_combined_summary.csv')
    contaminants_combined_summary.to_csv(contaminants_summary_path, index=False); print(f"  Resumen contaminantes guardado: {contaminants_summary_path}"); print(contaminants_combined_summary)

    print("\nGenerando resumen combinado de AQI...")
    df_aqi_only = df_all_detailed_metrics[df_all_detailed_metrics['Variable'] == 'AQI']
    aqi_combined_summary = df_aqi_only.groupby('City')[['MAE', 'RMSE', 'NRMAE_range', 'NRMSE_range']].mean().reset_index()
    aqi_combined_summary.rename(columns={'MAE': 'Avg_MAE_AQI', 'RMSE': 'Avg_RMSE_AQI', 'NRMAE_range': 'Avg_NRMAE_range_AQI', 'NRMSE_range': 'Avg_NRMSE_range_AQI'}, inplace=True)
    aqi_summary_path = os.path.join(METRICS_SARIMA_OUTPUT_DIR, 'metrics_aqi_combined_summary.csv')
    aqi_combined_summary.to_csv(aqi_summary_path, index=False); print(f"  Resumen AQI guardado: {aqi_summary_path}"); print(aqi_combined_summary)

    global_detailed_s_file = os.path.join(METRICS_SARIMA_OUTPUT_DIR, f'ALL_metrics_detailed_{SARIMA_METHOD_NAME}.csv')
    df_all_detailed_metrics.to_csv(global_detailed_s_file, index=False); print(f"\nMétricas detalladas globales de SARIMA guardadas en: {global_detailed_s_file}")
else:
    print("\nNo se generaron resultados de métricas detalladas.")

end_time_script_sarima = time.time()
print(f"\nProceso SARIMA completado. CSVs: {IMPUTED_CSVS_SARIMA_OUTPUT_DIR}, Métricas: {METRICS_SARIMA_OUTPUT_DIR}")
print(f"Tiempo SARIMA: {(end_time_script_sarima - start_time_script_sarima) / 60:.2f} min.")

Datos originales GT cargados desde: CSVs\Air_Quality.csv

Procesando Ciudad para Imputación SARIMA: Brasilia
  Escenario: RandomRows_20percent
    Imputando columna: CO...
    Imputando columna: NO2...
    Imputando columna: SO2...
    Imputando columna: O3...
    Imputando columna: PM2.5...
    Imputando columna: PM10...
    Imputando columna: AQI...
    Imputando columna: CO...
    Imputando columna: NO2...
    Imputando columna: SO2...
    Imputando columna: O3...
    Imputando columna: PM2.5...
    Imputando columna: PM10...
    Imputando columna: AQI...
    Imputando columna: CO...
    Imputando columna: NO2...
    Imputando columna: SO2...
    Imputando columna: O3...
    Imputando columna: PM2.5...
    Imputando columna: PM10...
    Imputando columna: AQI...
    Imputando columna: CO...
    Imputando columna: NO2...
    Imputando columna: SO2...
    Imputando columna: O3...
    Imputando columna: PM2.5...
    Imputando columna: PM10...
    Imputando columna: AQI...
    Imputando

RANDOM FOREST

In [2]:
import pandas as pd
import numpy as np
import os
import random
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
import time
import warnings
import glob

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# --- 0. Carga de Datos y Preparación Inicial ---
try:
    file_path_load = os.path.join('CSVs', 'Air_Quality.csv')
    if not os.path.exists(file_path_load):
        script_dir_parent = os.path.dirname(os.getcwd())
        alt_file_path_load = os.path.join(script_dir_parent, 'CSVs', 'Air_Quality.csv')
        if os.path.exists(alt_file_path_load): file_path_load = alt_file_path_load
        else: raise FileNotFoundError("Air_Quality.csv no encontrado.")
    df_air_quality_orig_load = pd.read_csv(file_path_load)
    print(f"Datos cargados desde: {file_path_load}")
except FileNotFoundError:
    print("Error: Air_Quality.csv no encontrado. Creando DataFrame de ejemplo...")
    _example_data = []
    _cities_example = ['Brasilia', 'Cairo', 'Dubai', 'London', 'New_York', 'Sydney']
    _start_date = pd.to_datetime('2024-01-01')
    for _city_ex in _cities_example:
        for i in range(200):
            _date = _start_date + pd.Timedelta(hours=i)
            _example_data.append([_date, _city_ex, 200+np.random.rand()*100, 20+np.random.rand()*10, 5+np.random.rand()*5, 50+np.random.rand()*30, 15+np.random.rand()*10, 30+np.random.rand()*20, 40+np.random.rand()*20])
    df_air_quality_orig_load = pd.DataFrame(_example_data, columns=['Date', 'City', 'CO', 'NO2', 'SO2', 'O3', 'PM2.5', 'PM10', 'AQI'])
    print("DataFrame de ejemplo creado.")

df_air_quality_orig_load['Date'] = pd.to_datetime(df_air_quality_orig_load['Date'])
if 'CO2' in df_air_quality_orig_load.columns:
    df_air_quality_orig_load = df_air_quality_orig_load.drop(columns=['CO2'])

# --- Parámetros ---
TARGET_CITY_RF = None # Poner a None para procesar todas las ciudades

CONTAMINANT_COLS = ['CO', 'NO2', 'SO2', 'O3', 'PM2.5', 'PM10']
AQI_COL = ['AQI']
ALL_TARGET_COLS = CONTAMINANT_COLS + AQI_COL

df_air_quality_orig_load['Hour'] = df_air_quality_orig_load['Date'].dt.hour
df_air_quality_orig_load['DayOfWeek'] = df_air_quality_orig_load['Date'].dt.dayofweek
df_air_quality_orig_load['Month'] = df_air_quality_orig_load['Date'].dt.month
TIME_FEATURES = ['Hour', 'DayOfWeek', 'Month']

N_LAGS = [1, 2, 24, 48]
LAG_FEATURES = []
df_with_features = df_air_quality_orig_load.copy()
for col_to_lag in ALL_TARGET_COLS:
    for lag_val in N_LAGS:
        lag_col_name = f"{col_to_lag}_lag{lag_val}"
        df_with_features[lag_col_name] = df_with_features.groupby('City')[col_to_lag].shift(lag_val)
        LAG_FEATURES.append(lag_col_name)

COLUMNS_FOR_IMPUTER_TRAINING_AND_TRANSFORM = ALL_TARGET_COLS + TIME_FEATURES + LAG_FEATURES

N_REPETITIONS = 5 # Ajustar según sea necesario
# CORRECCIÓN: Usar la nueva lista de escenarios
NAN_SIMULATION_SCENARIOS = [
    {'name': 'RandomRows_20percent'},
    {'name': 'BlockRows_12h_20percent'},
    {'name': 'BlockRows_24h_20percent'},
    {'name': 'BlockRows_36h_20percent'},
    {'name': 'BlockRows_48h_20percent'},
    {'name': 'BlockRows_60h_20percent'}
]

PROJECT_ROOT_DIR = os.getcwd()
SIMULATED_DATA_ROOT_DIR = os.path.join(PROJECT_ROOT_DIR, 'CSVs', 'simulados')

RF_IMPUTER_METHOD_NAME = 'RandomForestImputer_full'
IMPUTATIONS_RF_BASE_DIR = os.path.join(PROJECT_ROOT_DIR, 'imputaciones', RF_IMPUTER_METHOD_NAME)
METRICS_RF_OUTPUT_DIR = os.path.join(IMPUTATIONS_RF_BASE_DIR, 'metricas')
IMPUTED_CSVS_RF_OUTPUT_DIR = os.path.join(IMPUTATIONS_RF_BASE_DIR, 'CSVs')

os.makedirs(METRICS_RF_OUTPUT_DIR, exist_ok=True)
os.makedirs(IMPUTED_CSVS_RF_OUTPUT_DIR, exist_ok=True)

# --- Funciones Auxiliares (sin cambios) ---
def calculate_metrics_with_nmae_nrmse(y_true_df_reset, y_pred_df_reset, nan_mask_for_metrics_df_reset):
    results = {}
    for col in ALL_TARGET_COLS:
        results[col] = {'MAE': np.nan, 'RMSE': np.nan, 'NRMAE_range': np.nan, 'NRMSE_range': np.nan}
        if col not in y_true_df_reset.columns or col not in y_pred_df_reset.columns or col not in nan_mask_for_metrics_df_reset.columns: continue
        true_col_vals, pred_col_vals, mask_col_flags = y_true_df_reset[col].values, y_pred_df_reset[col].values, nan_mask_for_metrics_df_reset[col].values
        true_at_nans, pred_at_nans = true_col_vals[mask_col_flags], pred_col_vals[mask_col_flags]
        if len(true_at_nans) == 0: continue
        valid_pred_mask = ~np.isnan(pred_at_nans); true_at_nans_valid, pred_at_nans_valid = true_at_nans[valid_pred_mask], pred_at_nans[valid_pred_mask]
        if len(true_at_nans_valid) == 0: continue
        try:
            mae, rmse = mean_absolute_error(true_at_nans_valid, pred_at_nans_valid), np.sqrt(mean_squared_error(true_at_nans_valid, pred_at_nans_valid))
            original_col_full_series = y_true_df_reset[col].dropna()
            original_col_range = original_col_full_series.max() - original_col_full_series.min() if not original_col_full_series.empty else 0
            nrmae_range, nrmse_range = np.nan, np.nan
            if original_col_range > 1e-9: nrmae_range, nrmse_range = mae / original_col_range, rmse / original_col_range
            results[col].update({'MAE': mae, 'RMSE': rmse, 'NRMAE_range': nrmae_range, 'NRMSE_range': nrmse_range})
        except Exception: pass
    return results

# --- Script Principal ---
all_detailed_metrics_rf_results = []
script_start_time_rf = time.time()

if 'df_with_features' not in locals() or df_with_features.empty:
    print("Error: df_with_features no está definido. Saliendo."); exit()

# CORRECCIÓN: Lógica para seleccionar ciudades a procesar
if TARGET_CITY_RF:
    cities_to_process = [TARGET_CITY_RF]
    if TARGET_CITY_RF not in df_with_features['City'].unique():
        print(f"Advertencia: La ciudad objetivo '{TARGET_CITY_RF}' no se encuentra en el dataset. No se procesará nada.")
        cities_to_process = []
else:
    cities_to_process = df_with_features['City'].unique()

for city_name_original in cities_to_process:
    city_name_safe = city_name_original.replace(" ", "_").replace("-", "_")
    print(f"\nProcesando Ciudad para Imputación RF: {city_name_original}")

    df_city_original_features = df_with_features[df_with_features['City'] == city_name_original].copy()
    if df_city_original_features.empty: continue

    df_city_gt_eval = df_air_quality_orig_load[df_air_quality_orig_load['City'] == city_name_original].copy()
    if df_city_gt_eval[ALL_TARGET_COLS].isnull().any().any():
        for col_fill in ALL_TARGET_COLS:
             if col_fill in df_city_gt_eval.columns: df_city_gt_eval[col_fill] = df_city_gt_eval[col_fill].ffill().bfill()
    if df_city_gt_eval[ALL_TARGET_COLS].isnull().any().any():
        print(f"  Error: No se pudieron rellenar NaNs en el ground truth de {city_name_original}. Omitiendo."); continue
    df_city_original_ground_truth_reset = df_city_gt_eval.reset_index(drop=True)

    city_imputed_csv_rf_dir_root = os.path.join(IMPUTED_CSVS_RF_OUTPUT_DIR, city_name_safe)
    os.makedirs(city_imputed_csv_rf_dir_root, exist_ok=True)



    rf_estimator = RandomForestRegressor(n_estimators=50, random_state=42, n_jobs=-1, max_depth=15, min_samples_split=5, min_samples_leaf=2, max_features='sqrt')
    iterative_imputer_rf = IterativeImputer(estimator=rf_estimator, max_iter=15, tol=1e-3, random_state=42, initial_strategy='mean', verbose=1)
    scaler = StandardScaler()
    
    
    
    data_for_fitting_imputer = df_city_original_features[COLUMNS_FOR_IMPUTER_TRAINING_AND_TRANSFORM].copy()
    print(f"  Ajustando Scaler e IterativeImputer para {city_name_original}...")
    start_fit_time = time.time()
    try:
        scaled_data_for_fitting = scaler.fit_transform(data_for_fitting_imputer)
        iterative_imputer_rf.fit(scaled_data_for_fitting)
        print(f"  Scaler e Imputer ajustados en {time.time() - start_fit_time:.2f} segundos.")
    except Exception as e_fit:
        print(f"  Error ajustando Scaler/Imputer: {e_fit}. Omitiendo ciudad."); continue

    for scenario in NAN_SIMULATION_SCENARIOS:
        scenario_name_safe = scenario['name'].replace("%", "percent").replace(" ", "_")
        print(f"    Escenario: {scenario_name_safe}")
        
        sim_data_scenario_dir = os.path.join(SIMULATED_DATA_ROOT_DIR, city_name_safe, scenario_name_safe)
        imputed_rf_scenario_dir = os.path.join(city_imputed_csv_rf_dir_root, scenario_name_safe); os.makedirs(imputed_rf_scenario_dir, exist_ok=True)
        if not os.path.exists(sim_data_scenario_dir):
            print(f"      Directorio de datos simulados no encontrado: {sim_data_scenario_dir}. Omitiendo."); continue

        for rep in range(1, N_REPETITIONS + 1):
            matching_files = sorted(glob.glob(os.path.join(sim_data_scenario_dir, f"{city_name_safe}_{scenario_name_safe}_rep{rep}_*_with_nans.csv")))
            if not matching_files: continue
            sim_file_path_to_load = matching_files[0]
            current_seed_loaded = os.path.basename(sim_file_path_to_load).split('_seed')[-1].split('_with_nans.csv')[0]

            try:
                df_city_with_nans_loaded_raw = pd.read_csv(sim_file_path_to_load)
                df_for_imputation_base = df_city_original_features.copy().reset_index(drop=True)
                df_city_with_nans_loaded_reset = df_city_with_nans_loaded_raw.reset_index(drop=True)
                for col_target in ALL_TARGET_COLS:
                    if col_target in df_city_with_nans_loaded_reset.columns:
                        df_for_imputation_base[col_target] = df_city_with_nans_loaded_reset[col_target]
            except Exception as e_load:
                print(f"      Error cargando/procesando {sim_file_path_to_load}: {e_load}."); continue
            
            temp_introduced_nan_mask_df = pd.DataFrame(False, index=df_city_original_ground_truth_reset.index, columns=ALL_TARGET_COLS)
            for col_mask_check in ALL_TARGET_COLS:
                 if col_mask_check in df_city_with_nans_loaded_reset.columns and col_mask_check in df_city_original_ground_truth_reset.columns:
                    temp_introduced_nan_mask_df[col_mask_check] = df_city_with_nans_loaded_reset[col_mask_check].isnull() & ~df_city_original_ground_truth_reset[col_mask_check].isnull()

            df_to_impute_for_rf = df_for_imputation_base[COLUMNS_FOR_IMPUTER_TRAINING_AND_TRANSFORM].copy()
            
            try:
                scaled_data_to_impute = scaler.transform(df_to_impute_for_rf)
                imputed_scaled_data_rf_array = iterative_imputer_rf.transform(scaled_data_to_impute)
                imputed_data_rf_array = scaler.inverse_transform(imputed_scaled_data_rf_array)
                df_imputed_features = pd.DataFrame(imputed_data_rf_array, columns=COLUMNS_FOR_IMPUTER_TRAINING_AND_TRANSFORM)
                df_final_imputed_rf = df_for_imputation_base[['Date', 'City']].copy()
                for col_final in ALL_TARGET_COLS + TIME_FEATURES:
                    if col_final in df_imputed_features.columns: df_final_imputed_rf[col_final] = df_imputed_features[col_final]
            except Exception as e_impute:
                print(f"      Error durante la imputación RF para rep {rep}: {e_impute}."); continue

            imputed_filename_rf = f"{city_name_safe}_{scenario_name_safe}_rep{rep}_seed{current_seed_loaded}_imputed_rf.csv"
            df_final_imputed_rf.to_csv(os.path.join(imputed_rf_scenario_dir, imputed_filename_rf), index=False)

            metrics_scores_rf = calculate_metrics_with_nmae_nrmse(df_city_original_ground_truth_reset[ALL_TARGET_COLS], df_final_imputed_rf[ALL_TARGET_COLS], temp_introduced_nan_mask_df)
            for var_name in ALL_TARGET_COLS:
                all_detailed_metrics_rf_results.append({'City': city_name_original, 'Scenario': scenario_name_safe, 'Repetition': rep, 'Seed': current_seed_loaded, 'Imputation_Method': RF_IMPUTER_METHOD_NAME, 'Variable': var_name, 'MAE': metrics_scores_rf[var_name]['MAE'], 'RMSE': metrics_scores_rf[var_name]['RMSE'], 'NRMAE_range': metrics_scores_rf[var_name]['NRMAE_range'], 'NRMSE_range': metrics_scores_rf[var_name]['NRMSE_range']})
    print(f"  Procesamiento de escenarios RF para {city_name_original} completado.")

# --- Guardado de Métricas Resumidas ---
if all_detailed_metrics_rf_results:
    df_all_detailed_metrics_rf = pd.DataFrame(all_detailed_metrics_rf_results)
    print("\nGenerando resúmenes de métricas RF por variable y ciudad...")
    grouped_by_city_var_rf = df_all_detailed_metrics_rf.groupby(['City', 'Variable'])[['MAE', 'RMSE', 'NRMAE_range', 'NRMSE_range']].mean().reset_index()
    grouped_by_city_var_rf.rename(columns={'MAE': 'Avg_MAE', 'RMSE': 'Avg_RMSE', 'NRMAE_range': 'Avg_NRMAE_range', 'NRMSE_range': 'Avg_NRMSE_range'}, inplace=True)
    for city_name_original_summary in grouped_by_city_var_rf['City'].unique():
        city_name_safe_rf_summary = city_name_original_summary.replace(" ", "_").replace("-", "_"); city_metrics_df_rf = grouped_by_city_var_rf[grouped_by_city_var_rf['City'] == city_name_original_summary]
        variables_summary_path_rf = os.path.join(METRICS_RF_OUTPUT_DIR, f'metrics_variables_summary_{city_name_safe_rf_summary}.csv')
        city_metrics_df_rf.to_csv(variables_summary_path_rf, index=False); print(f"  Resumen RF de variables para {city_name_original_summary} guardado en: {variables_summary_path_rf}")
        if city_name_original_summary == cities_to_process[0]: print(city_metrics_df_rf)

    print("\nGenerando resumen RF combinado de contaminantes...")
    df_contaminants_only_rf = df_all_detailed_metrics_rf[df_all_detailed_metrics_rf['Variable'].isin(CONTAMINANT_COLS)]
    contaminants_combined_summary_rf = df_contaminants_only_rf.groupby('City')[['MAE', 'RMSE', 'NRMAE_range', 'NRMSE_range']].mean().reset_index()
    contaminants_combined_summary_rf.rename(columns={'MAE': 'Avg_MAE_Conts', 'RMSE': 'Avg_RMSE_Conts', 'NRMAE_range': 'Avg_NRMAE_range_Conts', 'NRMSE_range': 'Avg_NRMSE_range_Conts'}, inplace=True)
    contaminants_summary_path_rf = os.path.join(METRICS_RF_OUTPUT_DIR, 'metrics_contaminants_combined_summary.csv')
    contaminants_combined_summary_rf.to_csv(contaminants_summary_path_rf, index=False); print(f"  Resumen RF combinado de contaminantes guardado en: {contaminants_summary_path_rf}"); print(contaminants_combined_summary_rf)
    
    print("\nGenerando resumen RF combinado de AQI...")
    df_aqi_only_rf = df_all_detailed_metrics_rf[df_all_detailed_metrics_rf['Variable'] == 'AQI']
    aqi_combined_summary_rf = df_aqi_only_rf.groupby('City')[['MAE', 'RMSE', 'NRMAE_range', 'NRMSE_range']].mean().reset_index()
    aqi_combined_summary_rf.rename(columns={'MAE': 'Avg_MAE_AQI', 'RMSE': 'Avg_RMSE_AQI', 'NRMAE_range': 'Avg_NRMAE_range_AQI', 'NRMSE_range': 'Avg_NRMSE_range_AQI'}, inplace=True)
    aqi_summary_path_rf = os.path.join(METRICS_RF_OUTPUT_DIR, 'metrics_aqi_combined_summary.csv')
    aqi_combined_summary_rf.to_csv(aqi_summary_path_rf, index=False); print(f"  Resumen RF combinado de AQI guardado en: {aqi_summary_path_rf}"); print(aqi_combined_summary_rf)
else:
    print("No se generaron resultados de métricas detalladas para Random Forest Imputer.")

script_end_time_rf = time.time()
print(f"\nProceso de Imputación con Random Forest completado. Tiempo total: {(script_end_time_rf - script_start_time_rf) / 60:.2f} minutos.")

Datos cargados desde: CSVs\Air_Quality.csv

Procesando Ciudad para Imputación RF: Brasilia
  Ajustando Scaler e IterativeImputer para Brasilia...
[IterativeImputer] Completing matrix with shape (8784, 38)
[IterativeImputer] Change: 6.442078832853756, scaled tolerance: 0.008004094305323934 
[IterativeImputer] Change: 3.7969897000559465, scaled tolerance: 0.008004094305323934 
[IterativeImputer] Change: 2.484989515146828, scaled tolerance: 0.008004094305323934 
[IterativeImputer] Change: 3.0486658070600745, scaled tolerance: 0.008004094305323934 
[IterativeImputer] Change: 2.9925322609138094, scaled tolerance: 0.008004094305323934 
[IterativeImputer] Change: 2.453738293096787, scaled tolerance: 0.008004094305323934 
[IterativeImputer] Change: 2.349269439012306, scaled tolerance: 0.008004094305323934 
[IterativeImputer] Change: 1.6335229209468904, scaled tolerance: 0.008004094305323934 
[IterativeImputer] Change: 1.218716941220053, scaled tolerance: 0.008004094305323934 
[IterativeImputer

LSTM-VAE

TARGET_CITY_VAE = "Brasilia"
CONTAMINANT_COLS = ['CO', 'NO2', 'SO2', 'O3', 'PM2.5', 'PM10']
AQI_COL = ['AQI']
ALL_TARGET_COLS_VAE = CONTAMINANT_COLS + AQI_COL
N_LAGS_VAE = 60
LATENT_DIM_VAE = 16
ENCODER_NEURONS_L1_VAE = 64
DECODER_NEURONS_L1_VAE = 64
VAE_INITIAL_LR = 5e-4; KL_LOSS_WEIGHT = 0.00005
VAE_BATCH_SIZE = 32
WFV_INITIAL_TRAIN_POINTS_MIN_RATIO = 0.1
WFV_VALIDATION_POINTS_MIN_RATIO = 0.1
WFV_STEP_POINTS_MIN_RATIO = 0.1
WFV_EPOCHS_PER_FOLD = 30
WFV_ES_PATIENCE = 5
WFV_LR_PATIENCE = 3
MAX_WFV_FOLDS = 3
N_REPETITIONS_FOR_EVAL = 1
NAN_SIMULATION_SCENARIOS = [
    {'name': 'RandomRows_20percent'},
    {'name': 'BlockRows_12h_20percent'},
    {'name': 'BlockRows_24h_20percent'},
    {'name': 'BlockRows_36h_20percent'},
    {'name': 'BlockRows_48h_20percent'},
    {'name': 'BlockRows_60h_20percent'}
]



FUNCIONAAAAAA

In [25]:
import pandas as pd
import numpy as np
import os
import json
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
import tensorflow as tf
import keras # Keras 3 de alto nivel
from keras import layers, Model
from keras import backend as K
import time
import glob
import warnings
import shutil
import re
from unicodedata import normalize
import joblib
import random
import matplotlib.pyplot as plt
import seaborn as sns

"""
Este script ha sido modificado para corregir la causa raíz de los errores de NaN.
Los cambios principales son:
1.  **Uso de Variables Exógenas Correctas:** Durante la fase de imputación, en lugar de intentar
    reconstruir las características de tiempo (Hour, DayOfWeek, Month) desde el archivo con
    NaNs (lo que fallaba), ahora se utilizan las características de tiempo del DataFrame
    original y limpio (`df_city_clean`). El tiempo no tiene valores faltantes, por lo que
    esta es la fuente de datos correcta y robusta.
2.  **Se mantienen los diagnósticos:** Las comprobaciones de NaN en la función de imputación
    se mantienen como medida de seguridad, pero ya no deberían activarse.
"""

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# --- INICIO DE LA CONFIGURACIÓN DE GPU ---
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        logical_gpus = tf.config.experimental.list_logical_devices('GPU')
        print(f"GPU detectada: {len(gpus)} Physical GPUs, {len(logical_gpus)} Logical GPUs")
        print("TensorFlow utilizará la GPU para el entrenamiento.")
    except RuntimeError as e:
        print(e)
else:
    print("No se detectó ninguna GPU. TensorFlow se ejecutará en la CPU.")

# --- Helper Function for Sanitizing File/Path Names ---
def sanitize_name(name_str):
    s = str(name_str).strip()
    s = re.sub(r'[^\w\s-]', '', s)
    s = re.sub(r'[-\s]+', '_', s)
    return s if s else "default_sanitized_name"

def sanitize_tf_name(name_str):
    s = normalize('NFKD', str(name_str)); s = s.encode('ascii', 'ignore'); s = s.decode('ascii')
    s = re.sub(r'[^a-zA-Z0-9_]', '_', s)
    if s and s[0].isdigit() and not s.isdigit(): s = "n_" + s
    if not s: s = "sanitized_default_name"
    s = re.sub(r'_+', '_', s)
    if s.startswith('_') and len(s) > 1: s = s[1:]
    if s.endswith('_') and len(s) > 1: s = s[:-1]
    if not s: s = "sanitized_default_name_v2"
    return s

# --- 0. Carga de Datos Originales ---
try:
    file_path_load = os.path.join('CSVs', 'Air_Quality.csv')
    df_air_quality_raw = pd.read_csv(file_path_load)
    print(f"Datos originales cargados desde: {file_path_load}")
except FileNotFoundError:
    print("Error: Air_Quality.csv no encontrado.")
    exit()

df_air_quality_raw['Date'] = pd.to_datetime(df_air_quality_raw['Date'], utc=True)
if 'CO2' in df_air_quality_raw.columns:
    df_air_quality_raw = df_air_quality_raw.drop(columns=['CO2'])
df_air_quality_raw['City'] = df_air_quality_raw['City'].replace('New_York', 'New York')

# --- 1. Conversión de Zona Horaria y Extracción de Características de Tiempo ---
print("\nConvirtiendo fechas de UTC a la hora local de cada ciudad y extrayendo características de tiempo...")
city_timezones = {'Brasilia': 'America/Sao_Paulo', 'Cairo': 'Africa/Cairo', 'Dubai': 'Asia/Dubai','London': 'Europe/London', 'New York': 'America/New_York', 'Sydney': 'Australia/Sydney'}
df_parts = []
TIME_FEATURES = ['Hour', 'DayOfWeek', 'Month']
for city, tz in city_timezones.items():
    df_city_part = df_air_quality_raw[df_air_quality_raw['City'] == city].copy()
    if not df_city_part.empty:
        df_city_part['Date'] = df_city_part['Date'].dt.tz_convert(tz)
        df_city_part['Hour'] = df_city_part['Date'].dt.hour
        df_city_part['DayOfWeek'] = df_city_part['Date'].dt.dayofweek
        df_city_part['Month'] = df_city_part['Date'].dt.month
        df_parts.append(df_city_part)
if not df_parts: print("Advertencia: No se procesaron datos."); df_air_quality_full_original = pd.DataFrame(columns=list(df_air_quality_raw.columns) + TIME_FEATURES)
else: df_air_quality_full_original = pd.concat(df_parts).sort_values(by=['City', 'Date']).reset_index(drop=True)
print("Conversión de zona horaria y extracción de características de tiempo completada.")

# --- Parámetros ---
TARGET_CITY_VAE = None  # Cambiar a None para procesar todas las ciudades
ALL_TARGET_COLS_VAE = ['CO', 'NO2', 'SO2', 'O3', 'PM2.5', 'PM10', 'AQI']
N_LAGS_VAE = 60
LATENT_DIM_VAE, ENCODER_NEURONS_L1_VAE, DECODER_NEURONS_L1_VAE = 16, 64, 64
VAE_INITIAL_LR, KL_LOSS_WEIGHT = 5e-4, 0.00005
VAE_BATCH_SIZE = 64
WFV_INITIAL_TRAIN_POINTS_MIN_RATIO, WFV_VALIDATION_POINTS_MIN_RATIO, WFV_STEP_POINTS_MIN_RATIO = 0.2, 0.1, 0.1
WFV_EPOCHS_PER_FOLD, WFV_ES_PATIENCE, WFV_LR_PATIENCE = 80, 25, 10
MAX_WFV_FOLDS = 5
N_REPETITIONS_FOR_EVAL = 5
NAN_SIMULATION_SCENARIOS = [{'name': name} for name in ['RandomRows_20percent', 'BlockRows_12h_20percent', 'BlockRows_24h_20percent', 'BlockRows_36h_20percent', 'BlockRows_48h_20percent', 'BlockRows_60h_20percent']]

# --- Rutas ---
PROJECT_ROOT_DIR = os.getcwd()
SIMULATED_DATA_ROOT_DIR = os.path.join(PROJECT_ROOT_DIR, 'CSVs', 'simulados')
LSTM_VAE_METHOD_NAME = 'LSTM_VAE_full_final'
IMPUTATIONS_LSTM_VAE_BASE_DIR = os.path.join(PROJECT_ROOT_DIR, 'imputaciones', LSTM_VAE_METHOD_NAME)
METRICS_LSTM_VAE_OUTPUT_DIR = os.path.join(IMPUTATIONS_LSTM_VAE_BASE_DIR, 'metricas')
IMPUTED_CSVS_LSTM_VAE_OUTPUT_DIR = os.path.join(IMPUTATIONS_LSTM_VAE_BASE_DIR, 'CSVs_unified')
MODELS_LSTM_VAE_OUTPUT_DIR = os.path.join(IMPUTATIONS_LSTM_VAE_BASE_DIR, 'model_weights')
PLOTS_LSTM_VAE_OUTPUT_DIR = os.path.join(IMPUTATIONS_LSTM_VAE_BASE_DIR, 'plots')
os.makedirs(METRICS_LSTM_VAE_OUTPUT_DIR, exist_ok=True); os.makedirs(IMPUTED_CSVS_LSTM_VAE_OUTPUT_DIR, exist_ok=True)
os.makedirs(MODELS_LSTM_VAE_OUTPUT_DIR, exist_ok=True); os.makedirs(PLOTS_LSTM_VAE_OUTPUT_DIR, exist_ok=True)


# --- Funciones VAE y Auxiliares ---
def plot_imputation_results_segmented(original_series, imputed_series, nan_mask, variable_name, city_name, scenario_name, output_dir, n_segments=10):
    total_len = len(original_series)
    if total_len == 0: return
    segment_len = total_len // n_segments
    scenario_plot_dir = os.path.join(output_dir, scenario_name)
    os.makedirs(scenario_plot_dir, exist_ok=True)
    for i in range(n_segments):
        start = i * segment_len
        end = (i + 1) * segment_len if i < n_segments - 1 else total_len
        if start >= end: continue
        segment_original = original_series.iloc[start:end]
        segment_imputed = imputed_series.iloc[start:end]
        segment_mask = nan_mask.iloc[start:end]
        plt.figure(figsize=(20, 8))
        plt.plot(segment_original.index, segment_original, color='royalblue', label='Ground Truth', alpha=0.9, zorder=2, linewidth=1.5)
        plt.plot(segment_imputed.index, segment_imputed, color='gray', linestyle='--', label='Observed/Imputed Series', alpha=0.7, zorder=1)
        imputed_zone = segment_imputed.where(segment_mask)
        plt.plot(imputed_zone.index, imputed_zone, color='red', linewidth=2.5, label='Imputed Values', zorder=3)
        plt.title(f'Imputation: {variable_name} in {city_name} (Segment {i+1}/{n_segments})\nScenario: {scenario_name}')
        plt.xlabel('Time Index'); plt.ylabel(f'{variable_name} Value'); plt.legend()
        plt.grid(True, which='both', linestyle='--', linewidth=0.5); plt.tight_layout()
        plot_filename = f"imputation_{sanitize_name(scenario_name)}_{sanitize_name(variable_name)}_seg_{i+1}.png"
        plt.savefig(os.path.join(scenario_plot_dir, plot_filename), dpi=150); plt.close()
    print(f"    {n_segments} gráficos guardados para {variable_name} en: {scenario_plot_dir}")

def sampling(args):
    z_mean, z_log_var = args; batch = tf.shape(z_mean)[0]; dim = tf.shape(z_mean)[1]
    epsilon = tf.random.normal(shape=(batch, dim)); return z_mean + K.exp(0.5 * z_log_var) * epsilon

class VAE(Model):
    def __init__(self, encoder, decoder, kl_loss_weight, **kwargs):
        super().__init__(**kwargs)
        self.encoder, self.decoder, self.kl_loss_weight = encoder, decoder, kl_loss_weight
        self.total_loss_tracker = keras.metrics.Mean(name="total_loss")
        self.reconstruction_loss_tracker = keras.metrics.Mean(name="reconstruction_loss")
        self.kl_loss_tracker = keras.metrics.Mean(name="kl_loss")
        self.mse_loss_fn = keras.losses.MeanSquaredError()
    @property
    def metrics(self): return [self.total_loss_tracker, self.reconstruction_loss_tracker, self.kl_loss_tracker]
    def call(self, inputs): z_mean, _, z = self.encoder(inputs); return self.decoder(z)
    def train_step(self, data):
        x = data[0] if isinstance(data, tuple) else data
        with tf.GradientTape() as tape:
            z_mean, z_log_var, z = self.encoder(x); reconstruction = self.decoder(z)
            reconstruction_loss = self.mse_loss_fn(x[:, :, :1], reconstruction)
            kl_loss = -0.5 * K.mean(1 + z_log_var - K.square(z_mean) - K.exp(z_log_var))
            total_loss = reconstruction_loss + self.kl_loss_weight * kl_loss
        grads = tape.gradient(total_loss, self.trainable_weights); self.optimizer.apply_gradients(zip(grads, self.trainable_weights))
        self.total_loss_tracker.update_state(total_loss); self.reconstruction_loss_tracker.update_state(reconstruction_loss); self.kl_loss_tracker.update_state(kl_loss)
        return {m.name: m.result() for m in self.metrics}
    def test_step(self, data):
        x = data[0] if isinstance(data, tuple) else data
        z_mean, z_log_var, z = self.encoder(x); reconstruction = self.decoder(z)
        reconstruction_loss = self.mse_loss_fn(x[:, :, :1], reconstruction)
        kl_loss = -0.5 * K.mean(1 + z_log_var - K.square(z_mean) - K.exp(z_log_var))
        total_loss = reconstruction_loss + self.kl_loss_weight * kl_loss
        self.total_loss_tracker.update_state(total_loss); self.reconstruction_loss_tracker.update_state(reconstruction_loss); self.kl_loss_tracker.update_state(kl_loss)
        return {m.name: m.result() for m in self.metrics}

def create_vae_model_architecture(n_lags, n_features, latent_dim, enc_neurons, dec_neurons, var_name="var"):
    prefix = sanitize_tf_name(var_name)
    enc_in = layers.Input(shape=(n_lags, n_features), name=f"{prefix}_enc_in"); enc_lstm = layers.LSTM(enc_neurons, name=f"{prefix}_enc_lstm")(enc_in)
    z_m, z_lv = layers.Dense(latent_dim, name=f"{prefix}_z_mean")(enc_lstm), layers.Dense(latent_dim, name=f"{prefix}_z_log_var")(enc_lstm)
    z = layers.Lambda(sampling, name=f"{prefix}_z_samp")([z_m, z_lv]); encoder = Model(enc_in, [z_m, z_lv, z], name=f"{prefix}_encoder")
    dec_in = layers.Input(shape=(latent_dim,), name=f"{prefix}_dec_in"); dec_rep = layers.RepeatVector(n_lags, name=f"{prefix}_dec_rep")(dec_in)
    dec_lstm = layers.LSTM(dec_neurons, return_sequences=True, name=f"{prefix}_dec_lstm")(dec_rep)
    dec_out = layers.TimeDistributed(layers.Dense(1), name=f"{prefix}_dec_out")(dec_lstm); decoder = Model(dec_in, dec_out, name=f"{prefix}_decoder")
    return encoder, decoder

def create_sequences_for_vae_training(target_1d, exog_df, n_lags):
    X = []; n_exog = exog_df.shape[1] if exog_df is not None else 0
    for i in range(len(target_1d) - n_lags + 1):
        target_win = target_1d[i : i + n_lags].reshape(-1, 1)
        if n_exog > 0: X.append(np.concatenate((target_win, exog_df.iloc[i : i + n_lags].values), axis=1))
        else: X.append(target_win)
    return np.array(X) if X else np.array([]).reshape(0, n_lags, 1 + n_exog)

def impute_segment_with_lstm_vae_batch(target_series, exog_df_scaled, vae, scaler, n_lags, fallback_val, var_name_for_debug=""):
    imputed = pd.Series(target_series.copy().flatten())
    nan_indices = imputed[imputed.isnull()].index.tolist()

    if not nan_indices:
        return imputed.values.reshape(-1, 1)

    print(f"\n  [DEBUG][{var_name_for_debug}] Iniciando imputación. Total NaNs a procesar: {len(nan_indices)}")

    temp_filled = imputed.ffill().bfill().fillna(fallback_val)
    if temp_filled.isnull().any():
        print(f"  [CRITICAL_WARN][{var_name_for_debug}] 'temp_filled' todavía contiene NaNs después de ffill/bfill/fillna. Rellenando de nuevo.")
        temp_filled.fillna(fallback_val, inplace=True)

    sequences, indices_to_impute = [], []
    
    for i, idx in enumerate(nan_indices):
        if idx < n_lags - 1:
            continue
        
        start = idx - n_lags + 1
        end = idx + 1

        target_win_temp = temp_filled.iloc[start:end].values.reshape(-1, 1)
        target_win_scaled = scaler.transform(target_win_temp)
        
        if exog_df_scaled is not None:
            exog_win = exog_df_scaled.iloc[start:end].values
            seq = np.concatenate((target_win_scaled, exog_win), axis=1)
        else:
            seq = target_win_scaled
        
        if np.isnan(seq).any():
            print(f"  [ERROR][{var_name_for_debug}] ¡Se ha generado una secuencia con NaN en el índice {idx}! Se omitirá esta imputación.")
            continue

        sequences.append(seq)
        indices_to_impute.append(idx)

    if not sequences:
        print(f"  [WARN][{var_name_for_debug}] No se generaron secuencias válidas para imputar. Se devolverá la serie con ffill/bfill.")
        return temp_filled.values.reshape(-1, 1)
    
    print(f"  [INFO][{var_name_for_debug}] Se generaron {len(sequences)} secuencias válidas para la imputación.")
    sequences_array = np.array(sequences)    
    reconstructions = vae.predict(sequences_array, batch_size=512, verbose=0)
    
    imputed_scaled = reconstructions[:, -1, 0]
    if np.isnan(imputed_scaled).any():
        print(f"  [ERROR][{var_name_for_debug}] ¡La salida del modelo (reconstrucción escalada) contiene NaNs!")
    
    imputed_unscaled = scaler.inverse_transform(imputed_scaled.reshape(-1, 1)).flatten()

    variance = np.var(imputed_unscaled)
    print(f"  [INFO][{var_name_for_debug}] Varianza de los valores imputados (des-escalados): {variance:.6f}")
    if np.isnan(variance):
        nan_count = np.isnan(imputed_unscaled).sum()
        total_count = len(imputed_unscaled)
        print(f"  [ERROR][{var_name_for_debug}] La varianza es NaN. Recuento de NaNs en valores imputados: {nan_count} de {total_count}")
    elif variance < 1e-6:
        print(f"  [WARN][{var_name_for_debug}] La varianza es casi cero. Las imputaciones son planas.")
    
    imputed.loc[indices_to_impute] = imputed_unscaled
    return imputed.ffill().bfill().fillna(fallback_val).values.reshape(-1, 1)

def calculate_metrics_with_nmae_nrmse(y_true_df, y_pred_df, nan_mask_df, cols_to_eval):
    res = {}
    y_true, y_pred, mask = y_true_df.reset_index(drop=True), y_pred_df.reset_index(drop=True), nan_mask_df.reset_index(drop=True)
    for col in cols_to_eval:
        if col not in y_true.columns or col not in y_pred.columns: continue
        res[col] = {'MAE': np.nan, 'RMSE': np.nan, 'NRMAE_range': np.nan, 'NRMSE_range': np.nan}
        true_vals, pred_vals, m = y_true[col], y_pred[col], mask[col].astype(bool)
        if m.sum() == 0: continue
        try:
            mae, rmse = mean_absolute_error(true_vals[m], pred_vals[m]), np.sqrt(mean_squared_error(true_vals[m], pred_vals[m]))
            data_range = y_true[col].max() - y_true[col].min()
            res[col].update({'MAE': mae, 'RMSE': rmse, 'NRMAE_range': mae / data_range if data_range > 1e-9 else np.nan, 'NRMSE_range': rmse / data_range if data_range > 1e-9 else np.nan})
        except ValueError:
            print(f"  [METRICS_ERROR] ValueError al calcular métricas para {col}. Podría haber NaNs o Infs en los valores imputados.")
    return res

# --- Script Principal ---
all_detailed_metrics_lstm_vae_results = []
script_start_time_lstm_vae = time.time()
cities_to_process_vae = [TARGET_CITY_VAE] if TARGET_CITY_VAE else (df_air_quality_full_original['City'].unique() if not df_air_quality_full_original.empty else [])

for city in cities_to_process_vae:
    print(f"\n=== PROCESANDO CIUDAD: {city} ===")
    city_name_safe = sanitize_name(city)
    df_city_clean = df_air_quality_full_original[df_air_quality_full_original['City'] == city].copy()
    if df_city_clean.empty: continue
    
    trained_vaes, scalers_target = {}, {}
    df_exog_base = df_city_clean[TIME_FEATURES].ffill().bfill()
    if df_exog_base.isnull().any().any(): continue
    
    scaler_exog = MinMaxScaler()
    # Ajustar y transformar las variables exógenas del conjunto de datos limpio
    scaled_exog_df_clean = pd.DataFrame(scaler_exog.fit_transform(df_exog_base), columns=TIME_FEATURES)
    n_features = 1 + scaled_exog_df_clean.shape[1]
    
    city_model_dir = os.path.join(MODELS_LSTM_VAE_OUTPUT_DIR, city_name_safe); os.makedirs(city_model_dir, exist_ok=True)
    city_plots_dir = os.path.join(PLOTS_LSTM_VAE_OUTPUT_DIR, city_name_safe); os.makedirs(city_plots_dir, exist_ok=True)
    joblib.dump(scaler_exog, os.path.join(city_model_dir, f'scaler_exog_{city_name_safe}.joblib'))

    for var in ALL_TARGET_COLS_VAE:
        K.clear_session(); tf.random.set_seed(42); np.random.seed(42); random.seed(42)
        print(f"\n--- Preparando VAE para VARIABLE: {var} (Ciudad: {city}) ---")
        
        series_clean = df_city_clean[var].ffill().bfill()
        if series_clean.isnull().any(): continue
        
        scaler_target = MinMaxScaler(); scaled_target_full = scaler_target.fit_transform(series_clean.values.reshape(-1, 1))
        scalers_target[var] = scaler_target
        
        encoder, decoder = create_vae_model_architecture(N_LAGS_VAE, n_features, LATENT_DIM_VAE, ENCODER_NEURONS_L1_VAE, DECODER_NEURONS_L1_VAE, var)
        
        print(f"  Iniciando WFV para VAE de {var}...")
        total_points = len(scaled_target_full); init_train_len = int(total_points * WFV_INITIAL_TRAIN_POINTS_MIN_RATIO)
        val_len = int(total_points * WFV_VALIDATION_POINTS_MIN_RATIO); step_len = max(1, int(total_points * WFV_STEP_POINTS_MIN_RATIO))
        if total_points < init_train_len + val_len + N_LAGS_VAE: print("  Datos insuficientes para WFV."); trained_vaes[var] = None; continue

        current_train_end, fold, best_val_loss = init_train_len, 0, float('inf')
        model_weights_path = os.path.join(city_model_dir, f'vae_weights_best_{sanitize_tf_name(var)}.weights.h5')

        while (current_train_end + val_len <= total_points) and (fold < MAX_WFV_FOLDS):
            fold += 1
            train_target = scaled_target_full[:current_train_end]; val_target = scaled_target_full[current_train_end : current_train_end + val_len]
            train_exog, val_exog = scaled_exog_df_clean.iloc[:current_train_end], scaled_exog_df_clean.iloc[current_train_end : current_train_end + val_len]
            print(f"    WFV Fold {fold} ({var}): Train {len(train_target)}, Val {len(val_target)}")
            
            X_train = create_sequences_for_vae_training(train_target.flatten(), train_exog, N_LAGS_VAE)
            X_val = create_sequences_for_vae_training(val_target.flatten(), val_exog, N_LAGS_VAE)
            if X_train.shape[0] < VAE_BATCH_SIZE or X_val.shape[0] == 0: current_train_end += step_len; continue
            
            vae_model = VAE(encoder, decoder, KL_LOSS_WEIGHT)
            vae_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=VAE_INITIAL_LR), loss=None)
            if not vae_model.built: vae_model(tf.zeros((1, N_LAGS_VAE, n_features)))
            if fold > 1 and os.path.exists(model_weights_path):
                try: vae_model.load_weights(model_weights_path)
                except Exception as e: print(f"      Error cargando pesos en fold {fold}: {e}")

            callbacks = [keras.callbacks.EarlyStopping(monitor='val_total_loss', patience=WFV_ES_PATIENCE, restore_best_weights=True),
                         keras.callbacks.ReduceLROnPlateau(monitor='val_total_loss', patience=WFV_LR_PATIENCE, factor=0.5)]
            
            history = vae_model.fit(X_train, epochs=WFV_EPOCHS_PER_FOLD, batch_size=VAE_BATCH_SIZE, validation_data=(X_val, X_val), callbacks=callbacks, verbose=1)
            current_val_loss = min(history.history.get('val_total_loss', [float('inf')]))

            print(f"      Fold {fold} completado. Val Total Loss: {current_val_loss:.4f}")
            if current_val_loss < best_val_loss:
                best_val_loss = current_val_loss; print(f"      NUEVO MEJOR VAE ({var}, val_loss: {best_val_loss:.4f}). Guardando pesos.")
                vae_model.save_weights(model_weights_path)
            current_train_end += step_len; K.clear_session()
        
        final_model = VAE(encoder, decoder, KL_LOSS_WEIGHT)
        final_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=VAE_INITIAL_LR), loss=None)
        if not final_model.built: final_model(tf.zeros((1, N_LAGS_VAE, n_features)))
        if os.path.exists(model_weights_path):
            try: final_model.load_weights(model_weights_path); print(f"  Modelo VAE final para {var} cargado.")
            except Exception as e: print(f"  Error cargando pesos finales VAE {var}: {e}."); final_model = None
        else: print(f"  Pesos VAE no encontrados para {var}."); final_model = None
        trained_vaes[var] = final_model

    # --- Imputation Phase ---
    print(f"\n--- Iniciando Imputación para {city} ---")
    df_city_ground_truth_for_eval = df_city_clean.reset_index(drop=True)
    
    # Pre-escalar las características de tiempo UNA SOLA VEZ usando los datos limpios.
    # Este es el `scaled_exog_df` que se usará para TODAS las imputaciones.
    exog_for_impute_base = df_city_clean[TIME_FEATURES].ffill().bfill()
    scaled_exog_for_impute = pd.DataFrame(scaler_exog.transform(exog_for_impute_base), columns=TIME_FEATURES)

    for scenario in NAN_SIMULATION_SCENARIOS:
        scenario_name_for_dir = scenario['name']
        scenario_name_safe = sanitize_name(scenario['name'])
        print(f"  Procesando Escenario: {scenario_name_safe}")
        imputed_scenario_dir = os.path.join(IMPUTED_CSVS_LSTM_VAE_OUTPUT_DIR, city_name_safe, scenario_name_safe); os.makedirs(imputed_scenario_dir, exist_ok=True)
        
        for rep in range(1, N_REPETITIONS_FOR_EVAL + 1):
            sim_dir = os.path.join(SIMULATED_DATA_ROOT_DIR, city_name_safe, scenario['name'])
            sim_files = glob.glob(os.path.join(sim_dir, f"{city_name_safe}_{scenario['name']}_rep{rep}_*_with_nans.csv"))
            if not sim_files: continue
            
            df_with_nans = pd.read_csv(sim_files[0])
            df_imputed = df_with_nans.copy()
            nan_mask_eval = df_with_nans[ALL_TARGET_COLS_VAE].isnull() & ~df_city_ground_truth_for_eval[ALL_TARGET_COLS_VAE].isnull()
            
            # LA LÍNEA PROBLEMÁTICA HA SIDO ELIMINADA. USAMOS `scaled_exog_for_impute` DEFINIDO FUERA DEL BUCLE.
            
            for var in ALL_TARGET_COLS_VAE:
                vae, scaler_t = trained_vaes.get(var), scalers_target.get(var)
                if not vae or not scaler_t: 
                    df_imputed[var] = df_imputed[var].interpolate().ffill().bfill()
                    continue
                
                imputed_values = impute_segment_with_lstm_vae_batch(
                    df_with_nans[var].values, 
                    scaled_exog_for_impute, # Usamos la versión correcta y pre-calculada
                    vae, 
                    scaler_t, 
                    N_LAGS_VAE, 
                    df_city_clean[var].mean(),
                    var_name_for_debug=var
                )
                df_imputed[var] = imputed_values.flatten()
            
            imputed_filename = os.path.basename(sim_files[0]).replace("_with_nans.csv", "_imputed_vae_ALL.csv")
            df_imputed.to_csv(os.path.join(imputed_scenario_dir, imputed_filename), index=False)
            print(f"      Archivo imputado guardado: {imputed_filename}")

            metrics = calculate_metrics_with_nmae_nrmse(df_city_ground_truth_for_eval, df_imputed, nan_mask_eval, ALL_TARGET_COLS_VAE)
            for var_name, scores in metrics.items():
                all_detailed_metrics_lstm_vae_results.append({'City': city, 'Scenario': scenario_name_safe, 'Repetition': rep, 'Seed': 'N/A', 'Imputation_Method': LSTM_VAE_METHOD_NAME, 'Variable': var_name, **scores})
            
            if rep == 1:
                plot_imputation_results_segmented(
                    original_series=df_city_ground_truth_for_eval['AQI'],
                    imputed_series=df_imputed['AQI'],
                    nan_mask=nan_mask_eval['AQI'],
                    variable_name='AQI', city_name=city, scenario_name=scenario_name_safe,
                    output_dir=os.path.join(PLOTS_LSTM_VAE_OUTPUT_DIR, city_name_safe)
                )

if all_detailed_metrics_lstm_vae_results:
    df_metrics = pd.DataFrame(all_detailed_metrics_lstm_vae_results)
    sufijo = f"_{sanitize_name(TARGET_CITY_VAE)}" if TARGET_CITY_VAE else "_ALL_CITIES"
    df_metrics.to_csv(os.path.join(METRICS_LSTM_VAE_OUTPUT_DIR, f'ALL_metrics_detailed_{LSTM_VAE_METHOD_NAME}{sufijo}.csv'), index=False)
    print(f"\nMétricas detalladas guardadas.")
    
    print("\nGenerando resúmenes de métricas por variable y ciudad...")
    grouped_by_city_var = df_metrics.groupby(['City', 'Variable'])[['MAE', 'RMSE', 'NRMAE_range', 'NRMSE_range']].mean().reset_index()
    grouped_by_city_var.rename(columns={'MAE': 'Avg_MAE', 'RMSE': 'Avg_RMSE', 'NRMAE_range': 'Avg_NRMAE_range', 'NRMSE_range': 'Avg_NRMSE_range'}, inplace=True)
    for city_name_summary in grouped_by_city_var['City'].unique():
        city_name_safe_summary = sanitize_name(city_name_summary)
        city_metrics_df = grouped_by_city_var[grouped_by_city_var['City'] == city_name_summary]
        variables_summary_path = os.path.join(METRICS_LSTM_VAE_OUTPUT_DIR, f'metrics_variables_summary_{city_name_safe_summary}.csv')
        city_metrics_df.to_csv(variables_summary_path, index=False)
        print(f"  Resumen de variables para {city_name_summary} guardado en: {variables_summary_path}")

    print("\nGenerando resumen combinado de contaminantes...")
    CONTAMINANT_COLS_SUMMARY = ['CO', 'NO2', 'SO2', 'O3', 'PM2.5', 'PM10']
    df_contaminants_only = df_metrics[df_metrics['Variable'].isin(CONTAMINANT_COLS_SUMMARY)]
    if not df_contaminants_only.empty:
        contaminants_combined_summary = df_contaminants_only.groupby('City')[['MAE', 'RMSE', 'NRMAE_range', 'NRMSE_range']].mean().reset_index()
        contaminants_combined_summary.rename(columns={'MAE': 'Avg_MAE_Conts', 'RMSE': 'Avg_RMSE_Conts', 'NRMAE_range': 'Avg_NRMAE_range_Conts', 'NRMSE_range': 'Avg_NRMSE_range_Conts'}, inplace=True)
        contaminants_summary_path = os.path.join(METRICS_LSTM_VAE_OUTPUT_DIR, f'metrics_contaminants_combined_summary{sufijo}.csv')
        contaminants_combined_summary.to_csv(contaminants_summary_path, index=False)
        print(f"  Resumen contaminantes guardado: {contaminants_summary_path}")

    print("\nGenerando resumen combinado de AQI...")
    df_aqi_only = df_metrics[df_metrics['Variable'] == 'AQI']
    if not df_aqi_only.empty:
        aqi_combined_summary = df_aqi_only.groupby('City')[['MAE', 'RMSE', 'NRMAE_range', 'NRMSE_range']].mean().reset_index()
        aqi_combined_summary.rename(columns={'MAE': 'Avg_MAE_AQI', 'RMSE': 'Avg_RMSE_AQI', 'NRMAE_range': 'Avg_NRMAE_range_AQI', 'NRMSE_range': 'Avg_NRMSE_range_AQI'}, inplace=True)
        aqi_summary_path = os.path.join(METRICS_LSTM_VAE_OUTPUT_DIR, f'metrics_aqi_combined_summary{sufijo}.csv')
        aqi_combined_summary.to_csv(aqi_summary_path, index=False)
        print(f"  Resumen AQI guardado: {aqi_summary_path}")

print(f"\nProceso LSTM-VAE completado. Tiempo: {(time.time() - script_start_time_lstm_vae) / 60:.2f} min.")

GPU detectada: 1 Physical GPUs, 1 Logical GPUs
TensorFlow utilizará la GPU para el entrenamiento.
Datos originales cargados desde: CSVs\Air_Quality.csv

Convirtiendo fechas de UTC a la hora local de cada ciudad y extrayendo características de tiempo...
Conversión de zona horaria y extracción de características de tiempo completada.

=== PROCESANDO CIUDAD: Brasilia ===

--- Preparando VAE para VARIABLE: CO (Ciudad: Brasilia) ---
  Iniciando WFV para VAE de CO...
    WFV Fold 1 (CO): Train 1756, Val 878
Epoch 1/80
27/27 [==============================] - 4s 45ms/step - total_loss: 0.0606 - reconstruction_loss: 0.0606 - kl_loss: 0.0166 - val_total_loss: 0.0165 - val_reconstruction_loss: 0.0165 - val_kl_loss: 0.0412
Epoch 2/80
27/27 [==============================] - 0s 16ms/step - total_loss: 0.0114 - reconstruction_loss: 0.0114 - kl_loss: 0.0902 - val_total_loss: 0.0101 - val_reconstruction_loss: 0.0101 - val_kl_loss: 0.1826
Epoch 3/80
27/27 [==============================] - 0s 17ms/ste

MOMENT PRE CON PLOTS Y METRICAS

In [3]:
import pandas as pd
import numpy as np
import os
import torch
from torch.utils.data import DataLoader, Dataset
from momentfm import MOMENTPipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
from tqdm import tqdm
import joblib
import traceback
import time
import random
import warnings
import matplotlib.pyplot as plt
import seaborn as sns

"""
Este script realiza imputaciones con el modelo pre-entrenado MOMENT.
Los cambios incluyen:
1.  **Estilo de Gráfico Restaurado:** La función `plot_imputation_results_segmented` ha sido
    actualizada para mostrar la serie imputada como una línea roja continua sobre una
    línea gris discontinua, destacando visualmente las zonas de imputación.
2.  **Métricas Completas:** Se genera un conjunto completo de métricas de evaluación,
    incluyendo resúmenes por ciudad, contaminantes y AQI.
"""


warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# --- Configuración de Estilo para Gráficas ---
sns.set_theme(style="whitegrid")
# La función de ploteo ahora define su propio figsize
plt.rcParams['axes.titlesize'] = 18
plt.rcParams['axes.labelsize'] = 14
plt.rcParams['xtick.labelsize'] = 12
plt.rcParams['ytick.labelsize'] = 12
plt.rcParams['legend.fontsize'] = 14

# --- 0. Carga de Datos Originales (Ground Truth) ---
try:
    file_path_load_orig = os.path.join('CSVs', 'Air_Quality.csv')
    df_air_quality_ground_truth = pd.read_csv(file_path_load_orig)
    print(f"Datos originales (ground truth) cargados desde: {file_path_load_orig}")
except FileNotFoundError:
    print("Error: Air_Quality.csv no encontrado.")
    exit()

df_air_quality_ground_truth['Date'] = pd.to_datetime(df_air_quality_ground_truth['Date'])
if 'CO2' in df_air_quality_ground_truth.columns:
    df_air_quality_ground_truth = df_air_quality_ground_truth.drop(columns=['CO2'])

# --- Parámetros Configurables ---
TARGET_CITY_MOMENT =  None  # None para procesar todas las ciudades"
ALL_TARGET_COLS_MOMENT = ['CO', 'NO2', 'SO2', 'O3', 'PM2.5', 'PM10', 'AQI']
CONTAMINANT_COLS_MOMENT = ['CO', 'NO2', 'SO2', 'O3', 'PM2.5', 'PM10']
SEQ_LEN_MODEL_MOMENT = 512
IMPUTATION_STEP_MOMENT = SEQ_LEN_MODEL_MOMENT
BATCH_SIZE_IMPUTE_MOMENT = 16
MODEL_NAME_MOMENT_PRETRAINED = "AutonLab/MOMENT-1-large"
N_REPETITIONS_MOMENT_EVAL = 5
NAN_SIMULATION_SCENARIOS_MOMENT = [
    {'name': 'RandomRows_20percent'},
    {'name': 'BlockRows_12h_20percent'},
    {'name': 'BlockRows_24h_20percent'},
    {'name': 'BlockRows_36h_20percent'},
    {'name': 'BlockRows_48h_20percent'},
    {'name': 'BlockRows_60h_20percent'}
]

# --- Rutas ---
PROJECT_ROOT_DIR_MOMENT = os.getcwd()
SIMULATED_DATA_INPUT_ROOT_DIR_MOMENT = os.path.join(PROJECT_ROOT_DIR_MOMENT, 'CSVs', 'simulados')
MOMENT_METHOD_NAME = 'MOMENT_PRE_full_v2'
IMPUTATIONS_MOMENT_BASE_DIR = os.path.join(PROJECT_ROOT_DIR_MOMENT, 'imputaciones', MOMENT_METHOD_NAME)
METRICS_MOMENT_OUTPUT_DIR = os.path.join(IMPUTATIONS_MOMENT_BASE_DIR, 'metricas')
IMPUTED_CSVS_MOMENT_OUTPUT_DIR = os.path.join(IMPUTATIONS_MOMENT_BASE_DIR, 'CSVs')
SCALERS_MOMENT_OUTPUT_DIR = os.path.join(IMPUTATIONS_MOMENT_BASE_DIR, 'scalers')
PLOTS_MOMENT_OUTPUT_DIR = os.path.join(IMPUTATIONS_MOMENT_BASE_DIR, 'plots')
os.makedirs(METRICS_MOMENT_OUTPUT_DIR, exist_ok=True)
os.makedirs(IMPUTED_CSVS_MOMENT_OUTPUT_DIR, exist_ok=True)
os.makedirs(SCALERS_MOMENT_OUTPUT_DIR, exist_ok=True)
os.makedirs(PLOTS_MOMENT_OUTPUT_DIR, exist_ok=True)

def sanitize_name(name_str): # Helper para nombres de archivo
    import re
    s = str(name_str).strip().replace(" ", "_")
    s = re.sub(r'(?u)[^-\w.]', '', s)
    return s

# --- FUNCIÓN DE PLOTEO RESTAURADA ---
def plot_imputation_results_segmented(original_series, imputed_series, nan_mask, variable_name, city_name, scenario_name, output_dir, n_segments=10):
    total_len = len(original_series)
    if total_len == 0:
        print("  Advertencia: No se pueden generar gráficos para una serie vacía.")
        return
    
    segment_len = total_len // n_segments
    scenario_plot_dir = os.path.join(output_dir, scenario_name)
    os.makedirs(scenario_plot_dir, exist_ok=True)

    for i in range(n_segments):
        start = i * segment_len
        end = (i + 1) * segment_len if i < n_segments - 1 else total_len
        if start >= end: continue

        segment_original = original_series.iloc[start:end]
        segment_imputed = imputed_series.iloc[start:end]
        segment_mask = nan_mask.iloc[start:end]
        
        plt.figure(figsize=(20, 8))
        
        # 1. Graficar el "Ground Truth" en azul
        plt.plot(segment_original.index, segment_original, color='royalblue', label='Ground Truth', alpha=0.9, zorder=2, linewidth=1.5)
        
        # 2. Graficar la serie completa (observada + imputada) en gris discontinuo
        plt.plot(segment_imputed.index, segment_imputed, color='gray', linestyle='--', label='Observed/Imputed Series', alpha=0.7, zorder=1)
        
        # 3. Superponer la línea roja solo en las zonas imputadas
        imputed_zone = segment_imputed.where(segment_mask)
        plt.plot(imputed_zone.index, imputed_zone, color='red', linewidth=2.5, label='Imputed Values', zorder=3)
        
        plt.title(f'Imputation: {variable_name} in {city_name} (Segment {i+1}/{n_segments})\nScenario: {scenario_name}')
        plt.xlabel('Time Index')
        plt.ylabel(f'{variable_name} Value')
        plt.legend()
        plt.grid(True, which='both', linestyle='--', linewidth=0.5)
        plt.tight_layout()
        
        plot_filename = f"imputation_{sanitize_name(scenario_name)}_{sanitize_name(variable_name)}_seg_{i+1}.png"
        plt.savefig(os.path.join(scenario_plot_dir, plot_filename), dpi=150)
        plt.close() # Cerrar la figura para liberar memoria
        
    print(f"    {n_segments} gráficos guardados para {variable_name} en: {scenario_plot_dir}")


# --- Clases y Funciones Auxiliares ---
class MomentImputationDataset(Dataset):
    def __init__(self, data_array_1d, nan_mask_1d, seq_len, step):
        self.data_array, self.nan_mask, self.seq_len, self.step = data_array_1d.squeeze(), nan_mask_1d.squeeze(), seq_len, step
        self.indices = list(range(0, len(self.data_array) - self.seq_len + 1, self.step))
        if not self.indices or (len(self.data_array) > self.seq_len and self.indices[-1] < len(self.data_array) - self.seq_len):
            self.indices.append(len(self.data_array) - self.seq_len)
        if len(self.data_array) < self.seq_len: self.indices = [0]
    def __len__(self): return len(self.indices)
    def __getitem__(self, idx):
        start_idx = self.indices[idx];
        seq_data = self.data_array[start_idx : start_idx + self.seq_len];
        mask_data = self.nan_mask[start_idx : start_idx + self.seq_len]
        if len(seq_data) < self.seq_len:
            pad_len = self.seq_len - len(seq_data)
            seq_data = np.pad(seq_data, (0, pad_len), 'constant', constant_values=0)
            mask_data = np.pad(mask_data, (0, pad_len), 'constant', constant_values=True)
        return {'timeseries': torch.tensor(seq_data, dtype=torch.float32).unsqueeze(0), 'input_mask': torch.tensor(~mask_data, dtype=torch.bool)}

def impute_univar_with_moment_pipeline(df_univar_with_nans_scaled_zeros, original_nan_mask_univar, moment_pipeline, device, scaler, seq_len, step, batch_size):
    dataset = MomentImputationDataset(df_univar_with_nans_scaled_zeros.values, original_nan_mask_univar.values, seq_len, step)
    if len(dataset) == 0: return None
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    
    n_rows = len(df_univar_with_nans_scaled_zeros); accumulated_preds, accumulated_counts = np.zeros(n_rows), np.zeros(n_rows)
    moment_pipeline.eval()
    with torch.no_grad():
        for batch_idx, batch in enumerate(loader):
            timeseries, input_mask = batch['timeseries'].to(device), batch['input_mask'].to(device)
            try:
                outputs = moment_pipeline(x_enc=timeseries, input_mask=input_mask); preds = outputs.reconstruction.cpu().numpy().squeeze(1)
            except Exception as e: print(f"Error en batch {batch_idx}: {e}"); continue
            start_ds_idx = batch_idx * batch_size
            for i, pred_seq in enumerate(preds):
                current_ds_idx = start_ds_idx + i; 
                if current_ds_idx >= len(dataset.indices): continue
                start_row_idx = dataset.indices[current_ds_idx]; end_row_idx = min(start_row_idx + seq_len, n_rows); len_to_add = end_row_idx - start_row_idx
                accumulated_preds[start_row_idx:end_row_idx] += pred_seq[:len_to_add]; accumulated_counts[start_row_idx:end_row_idx] += 1
    accumulated_counts[accumulated_counts == 0] = 1
    imputed_series_scaled = accumulated_preds / accumulated_counts
    imputed_series_unscaled = scaler.inverse_transform(imputed_series_scaled.reshape(-1, 1)).flatten()
    final_series = df_univar_with_nans_scaled_zeros.copy().iloc[:, 0]
    final_series = pd.Series(scaler.inverse_transform(final_series.values.reshape(-1, 1)).flatten(), index=final_series.index)
    nan_mask = original_nan_mask_univar.iloc[:, 0]
    final_series[nan_mask] = imputed_series_unscaled[nan_mask]
    return final_series

def calculate_metrics_moment(y_true_df, y_pred_df, nan_mask_df, cols_to_eval):
    results = {}
    y_true, y_pred, mask = y_true_df.reset_index(drop=True), y_pred_df.reset_index(drop=True), nan_mask_df.reset_index(drop=True)
    for col in cols_to_eval:
        if col not in y_true.columns or col not in y_pred.columns: continue
        results[col] = {'MAE': np.nan, 'RMSE': np.nan, 'NRMAE_range': np.nan, 'NRMSE_range': np.nan}
        true_vals, pred_vals, m = y_true[col], y_pred[col], mask[col].astype(bool)
        if m.sum() == 0: continue
        true_at_nans, pred_at_nans = true_vals[m], pred_vals[m]
        mae, rmse = mean_absolute_error(true_at_nans, pred_at_nans), np.sqrt(mean_squared_error(true_at_nans, pred_at_nans))
        data_range = y_true[col].max() - y_true[col].min()
        results[col].update({'MAE': mae, 'RMSE': rmse, 'NRMAE_range': mae / data_range if data_range > 1e-9 else np.nan, 'NRMSE_range': rmse / data_range if data_range > 1e-9 else np.nan})
    return results

# --- Script Principal ---
all_detailed_metrics_moment_results = []
script_start_time_moment = time.time()
device = "cuda" if torch.cuda.is_available() else "cpu"; print(f"Usando dispositivo: {device}")
try:
    moment_pipeline_global = MOMENTPipeline.from_pretrained(MODEL_NAME_MOMENT_PRETRAINED, model_kwargs={'task_name': 'reconstruction'})
    moment_pipeline_global.init(); moment_pipeline_global = moment_pipeline_global.to(device)
    print("Modelo MOMENT pre-entrenado cargado, INICIALIZADO y movido al dispositivo.")
except Exception as e: print(f"Error crítico cargando modelo MOMENT: {e}"); traceback.print_exc(); exit()

cities_to_process = [TARGET_CITY_MOMENT] if TARGET_CITY_MOMENT else df_air_quality_ground_truth['City'].unique()

for city in cities_to_process:
    print(f"\n===== PROCESANDO CIUDAD: {city} =====")
    df_city_ground_truth_moment = df_air_quality_ground_truth[df_air_quality_ground_truth['City'] == city].copy()
    if df_city_ground_truth_moment.empty: continue
    for col in ALL_TARGET_COLS_MOMENT: df_city_ground_truth_moment[col] = df_city_ground_truth_moment[col].ffill().bfill()
    df_city_ground_truth_moment_reset = df_city_ground_truth_moment.reset_index(drop=True)
    
    scalers_for_city = {}
    city_scalers_dir = os.path.join(SCALERS_MOMENT_OUTPUT_DIR, city.replace(" ", "_")); os.makedirs(city_scalers_dir, exist_ok=True)
    for col_scale in ALL_TARGET_COLS_MOMENT:
        scaler = StandardScaler(); scaler.fit(df_city_ground_truth_moment_reset[[col_scale]].dropna())
        scalers_for_city[col_scale] = scaler
        joblib.dump(scaler, os.path.join(city_scalers_dir, f'scaler_{col_scale}.joblib'))
    
    city_name_safe = city.replace(" ", "_")
    city_imputed_csv_dir = os.path.join(IMPUTED_CSVS_MOMENT_OUTPUT_DIR, city_name_safe); os.makedirs(city_imputed_csv_dir, exist_ok=True)
    city_plots_dir = os.path.join(PLOTS_MOMENT_OUTPUT_DIR, city_name_safe); os.makedirs(city_plots_dir, exist_ok=True)
    
    for scenario in NAN_SIMULATION_SCENARIOS_MOMENT:
        scenario_name_safe = scenario['name']
        print(f"\n--- Procesando Escenario: {scenario_name_safe} ---")
        sim_data_dir = os.path.join(SIMULATED_DATA_INPUT_ROOT_DIR_MOMENT, city_name_safe, scenario_name_safe)
        imputed_scenario_dir = os.path.join(city_imputed_csv_dir, scenario_name_safe); os.makedirs(imputed_scenario_dir, exist_ok=True)
        if not os.path.exists(sim_data_dir): continue
        
        for rep in range(1, N_REPETITIONS_MOMENT_EVAL + 1):
            sim_files = [f for f in os.listdir(sim_data_dir) if f.startswith(f"{city_name_safe}_{scenario_name_safe}_rep{rep}") and f.endswith("_with_nans.csv")]
            if not sim_files: continue
            sim_file_path = os.path.join(sim_data_dir, sim_files[0])
            seed = sim_files[0].split('_seed')[-1].split('_with_nans.csv')[0]
            df_with_nans = pd.read_csv(sim_file_path)
            
            df_imputed_all_cols = df_with_nans.copy()
            nan_mask_original = df_with_nans[ALL_TARGET_COLS_MOMENT].isnull()
            
            for col_to_impute in tqdm(ALL_TARGET_COLS_MOMENT, desc=f"  Imputando Rep {rep}/{N_REPETITIONS_MOMENT_EVAL}", unit="var"):
                scaler_current = scalers_for_city[col_to_impute]
                series_to_impute = df_with_nans[[col_to_impute]]
                scaled_data = scaler_current.transform(series_to_impute.values)
                scaled_data[series_to_impute.isnull().values] = 0.0
                df_scaled_univar = pd.DataFrame(scaled_data, columns=[col_to_impute], index=series_to_impute.index)
                
                if moment_pipeline_global is None: continue
                imputed_series = impute_univar_with_moment_pipeline(df_scaled_univar, series_to_impute.isnull(), moment_pipeline_global, device, scaler_current, SEQ_LEN_MODEL_MOMENT, IMPUTATION_STEP_MOMENT, BATCH_SIZE_IMPUTE_MOMENT)
                if imputed_series is not None:
                    df_imputed_all_cols[col_to_impute] = imputed_series.values

            df_imputed_all_cols = df_imputed_all_cols.ffill().bfill()
            imputed_filename = f"{city_name_safe}_{scenario_name_safe}_rep{rep}_seed{seed}_imputed_momentALL.csv"
            df_imputed_all_cols.to_csv(os.path.join(imputed_scenario_dir, imputed_filename), index=False)
            print(f"    Archivo imputado guardado: {imputed_filename}")

            nan_mask_for_eval = df_with_nans[ALL_TARGET_COLS_MOMENT].isnull() & ~df_city_ground_truth_moment_reset[ALL_TARGET_COLS_MOMENT].isnull()
            metrics = calculate_metrics_moment(df_city_ground_truth_moment_reset, df_imputed_all_cols, nan_mask_for_eval, ALL_TARGET_COLS_MOMENT)
            for var, scores in metrics.items():
                all_detailed_metrics_moment_results.append({'City': city, 'Scenario': scenario_name_safe, 'Repetition': rep, 'Seed': seed, 'Imputation_Method': MOMENT_METHOD_NAME, 'Variable': var, **scores})
            print(f"    Métricas calculadas para rep {rep}")

            if rep == 1:
                plot_imputation_results_segmented(
                    original_series=df_city_ground_truth_moment_reset['AQI'],
                    imputed_series=df_imputed_all_cols['AQI'],
                    nan_mask=nan_mask_original['AQI'],
                    variable_name='AQI', city_name=city, scenario_name=scenario_name_safe,
                    output_dir=city_plots_dir
                )
        print(f"  Escenario {scenario_name_safe} completado.")

# --- SECCIÓN DE GUARDADO DE MÉTRICAS MEJORADA ---
if all_detailed_metrics_moment_results:
    df_metrics = pd.DataFrame(all_detailed_metrics_moment_results)
    sufijo = f"_{TARGET_CITY_MOMENT.replace(' ', '_')}" if TARGET_CITY_MOMENT else "_ALL_CITIES"
    
    # 1. Guardar las métricas detalladas completas
    detailed_path = os.path.join(METRICS_MOMENT_OUTPUT_DIR, f'ALL_metrics_detailed_{MOMENT_METHOD_NAME}{sufijo}.csv')
    df_metrics.to_csv(detailed_path, index=False)
    print(f"\n--- Resumen de Métricas MOMENT ---")
    print(f"Métricas detalladas completas guardadas en: {detailed_path}")

    # 2. Generar resúmenes por ciudad y variable, y guardar un archivo por ciudad
    print("\nGenerando resúmenes de métricas por variable y ciudad...")
    grouped_by_city_var = df_metrics.groupby(['City', 'Variable'])[['MAE', 'RMSE', 'NRMAE_range', 'NRMSE_range']].mean().reset_index()
    grouped_by_city_var.rename(columns={'MAE': 'Avg_MAE', 'RMSE': 'Avg_RMSE', 'NRMAE_range': 'Avg_NRMAE_range', 'NRMSE_range': 'Avg_NRMSE_range'}, inplace=True)
    
    for city_name_summary in grouped_by_city_var['City'].unique():
        city_name_safe_summary = city_name_summary.replace(" ", "_")
        city_metrics_df = grouped_by_city_var[grouped_by_city_var['City'] == city_name_summary]
        variables_summary_path = os.path.join(METRICS_MOMENT_OUTPUT_DIR, f'metrics_variables_summary_{city_name_safe_summary}.csv')
        city_metrics_df.to_csv(variables_summary_path, index=False)
        print(f"  Resumen de variables para {city_name_summary} guardado en: {variables_summary_path}")

    # 3. Generar resumen combinado de contaminantes para todas las ciudades
    print("\nGenerando resumen combinado de contaminantes...")
    df_contaminants_only = df_metrics[df_metrics['Variable'].isin(CONTAMINANT_COLS_MOMENT)]
    if not df_contaminants_only.empty:
        contaminants_combined_summary = df_contaminants_only.groupby('City')[['MAE', 'RMSE', 'NRMAE_range', 'NRMSE_range']].mean().reset_index()
        contaminants_combined_summary.rename(columns={'MAE': 'Avg_MAE_Conts', 'RMSE': 'Avg_RMSE_Conts', 'NRMAE_range': 'Avg_NRMAE_range_Conts', 'NRMSE_range': 'Avg_NRMSE_range_Conts'}, inplace=True)
        contaminants_summary_path = os.path.join(METRICS_MOMENT_OUTPUT_DIR, f'metrics_contaminants_combined_summary{sufijo}.csv')
        contaminants_combined_summary.to_csv(contaminants_summary_path, index=False)
        print(f"  Resumen de contaminantes guardado: {contaminants_summary_path}")
        print(contaminants_combined_summary)

    # 4. Generar resumen combinado de AQI para todas las ciudades
    print("\nGenerando resumen combinado de AQI...")
    df_aqi_only = df_metrics[df_metrics['Variable'] == 'AQI']
    if not df_aqi_only.empty:
        aqi_combined_summary = df_aqi_only.groupby('City')[['MAE', 'RMSE', 'NRMAE_range', 'NRMSE_range']].mean().reset_index()
        aqi_combined_summary.rename(columns={'MAE': 'Avg_MAE_AQI', 'RMSE': 'Avg_RMSE_AQI', 'NRMAE_range': 'Avg_NRMAE_range_AQI', 'NRMSE_range': 'Avg_NRMSE_range_AQI'}, inplace=True)
        aqi_summary_path = os.path.join(METRICS_MOMENT_OUTPUT_DIR, f'metrics_aqi_combined_summary{sufijo}.csv')
        aqi_combined_summary.to_csv(aqi_summary_path, index=False)
        print(f"  Resumen de AQI guardado: {aqi_summary_path}")
        print(aqi_combined_summary)

print(f"\nProceso de Imputación con MOMENT Preentrenado completado. Tiempo total: {(time.time() - script_start_time_moment) / 60:.2f} minutos.")

Datos originales (ground truth) cargados desde: CSVs\Air_Quality.csv
Usando dispositivo: cuda
Modelo MOMENT pre-entrenado cargado, INICIALIZADO y movido al dispositivo.

===== PROCESANDO CIUDAD: Brasilia =====

--- Procesando Escenario: RandomRows_20percent ---


  Imputando Rep 1/5: 100%|██████████| 7/7 [00:01<00:00,  5.33var/s]


    Archivo imputado guardado: Brasilia_RandomRows_20percent_rep1_seed1202032594_imputed_momentALL.csv
    Métricas calculadas para rep 1
    10 gráficos guardados para AQI en: c:\Users\felix\Documents\MÁSTER\TFM\TFM_Ambiental\AIR QUALITY\imputaciones\MOMENT_PRE_full_v2\plots\Brasilia\RandomRows_20percent


  Imputando Rep 2/5: 100%|██████████| 7/7 [00:01<00:00,  4.61var/s]


    Archivo imputado guardado: Brasilia_RandomRows_20percent_rep2_seed1202079186_imputed_momentALL.csv
    Métricas calculadas para rep 2


...

    Archivo imputado guardado: Sydney_BlockRows_60h_20percent_rep3_seed2388496402_imputed_momentALL.csv
    Métricas calculadas para rep 3


  Imputando Rep 4/5: 100%|██████████| 7/7 [00:01<00:00,  5.36var/s]


    Archivo imputado guardado: Sydney_BlockRows_60h_20percent_rep4_seed2388496402_imputed_momentALL.csv
    Métricas calculadas para rep 4


  Imputando Rep 5/5: 100%|██████████| 7/7 [00:01<00:00,  5.21var/s]

    Archivo imputado guardado: Sydney_BlockRows_60h_20percent_rep5_seed2388496402_imputed_momentALL.csv
    Métricas calculadas para rep 5
  Escenario BlockRows_60h_20percent completado.

--- Resumen de Métricas MOMENT ---
Métricas detalladas completas guardadas en: c:\Users\felix\Documents\MÁSTER\TFM\TFM_Ambiental\AIR QUALITY\imputaciones\MOMENT_PRE_full_v2\metricas\ALL_metrics_detailed_MOMENT_PRE_full_v2_ALL_CITIES.csv

Generando resúmenes de métricas por variable y ciudad...
  Resumen de variables para Brasilia guardado en: c:\Users\felix\Documents\MÁSTER\TFM\TFM_Ambiental\AIR QUALITY\imputaciones\MOMENT_PRE_full_v2\metricas\metrics_variables_summary_Brasilia.csv
  Resumen de variables para Cairo guardado en: c:\Users\felix\Documents\MÁSTER\TFM\TFM_Ambiental\AIR QUALITY\imputaciones\MOMENT_PRE_full_v2\metricas\metrics_variables_summary_Cairo.csv
  Resumen de variables para Dubai guardado en: c:\Users\felix\Documents\MÁSTER\TFM\TFM_Ambiental\AIR QUALITY\imputaciones\MOMENT_PRE_full_v

MOMENT FT FINAL

In [ ]:
import pandas as pd
import numpy as np
import os
import torch
from torch.utils.data import DataLoader, Dataset
import torch.nn as nn
from torch.optim.lr_scheduler import ReduceLROnPlateau
from tqdm import tqdm
import joblib
import traceback
import time
import random
import warnings
import shutil
import glob
import sys
import matplotlib.pyplot as plt
import seaborn as sns

# Attempt to import MOMENT, handle if not found
try:
    from momentfm import MOMENTPipeline
except ImportError:
    print("MOMENTfm library not found. Please install it: pip install momentfm")
    exit()

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

warnings.filterwarnings("ignore")

# --- Configuración de Estilo para Gráficas ---
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (20, 8)
plt.rcParams['axes.titlesize'] = 18
plt.rcParams['axes.labelsize'] = 14
plt.rcParams['xtick.labelsize'] = 12
plt.rcParams['ytick.labelsize'] = 12
plt.rcParams['legend.fontsize'] = 14

# --- Helper Function for Sanitizing File/Path Names ---
def sanitize_name(name_str):
    import re
    s = str(name_str)
    s = re.sub(r'[^\w\s-]', '', s).strip()
    s = re.sub(r'[-\s]+', '_', s)
    s = re.sub(r'_+', '_', s)
    return s if s else "default_sanitized_name"

# --- Parámetros Configurables ---
TARGET_CITY_FT = None # Poner a None para procesar todas las ciudades
CONTAMINANT_COLS_FT = ['CO']#, 'NO2', 'SO2', 'O3', 'PM2.5', 'PM10']
AQI_COL_FT = ['AQI']
ALL_TARGET_COLS_FT = CONTAMINANT_COLS_FT + AQI_COL_FT
SEQ_LEN_MODEL_FT = 512
IMPUTATION_STEP_FT = SEQ_LEN_MODEL_FT
BATCH_SIZE_IMPUTE_FT = 128
MODEL_NAME_PRETRAINED_FT = "AutonLab/MOMENT-1-large"
EXPECTED_MODEL_CHANNELS = 8

# Fine-tuning Parameters
FINETUNE_EPOCHS = 30
FINETUNE_TRAIN_RATIO_NEW = 0.8
FINETUNE_STEP_DATALOADER_NEW = 128
BATCH_SIZE_FINETUNE_NEW = 32
LEARNING_RATE_FINETUNE_NEW = 1e-5
SIMULATED_MISSING_RATIO_FOR_FT = 0.25
PATIENCE_EARLY_STOPPING_NEW = 15
LR_PATIENCE_NEW = 5
MIN_DELTA_EARLY_STOPPING_NEW = 1e-5

N_REPETITIONS_FT_EVAL = 5
NAN_SIMULATION_SCENARIOS_FT = [{'name': name} for name in ['RandomRows_20percent']]#, 'BlockRows_12h_20percent', 'BlockRows_24h_20percent', 'BlockRows_36h_20percent', 'BlockRows_48h_20percent', 'BlockRows_60h_20percent']]

# --- Rutas ---
PROJECT_ROOT_DIR_FT = os.getcwd()
SIMULATED_DATA_INPUT_ROOT_DIR_FT = os.path.join(PROJECT_ROOT_DIR_FT, 'CSVs', 'simulados')
MOMENT_METHOD_NAME_FT = f'MOMENT_FT_final_{sanitize_name(MODEL_NAME_PRETRAINED_FT.split("/")[-1])}'
IMPUTATIONS_FT_BASE_DIR = os.path.join(PROJECT_ROOT_DIR_FT, 'imputaciones', MOMENT_METHOD_NAME_FT)
METRICS_FT_OUTPUT_DIR = os.path.join(IMPUTATIONS_FT_BASE_DIR, 'metricas')
IMPUTED_CSVS_FT_OUTPUT_DIR = os.path.join(IMPUTATIONS_FT_BASE_DIR, 'CSVs')
MODELS_FT_OUTPUT_DIR = os.path.join(IMPUTATIONS_FT_BASE_DIR, 'modelos_ft_sim_impute')
SCALERS_FT_OUTPUT_DIR = os.path.join(IMPUTATIONS_FT_BASE_DIR, 'scalers')
PLOTS_FT_OUTPUT_DIR = os.path.join(IMPUTATIONS_FT_BASE_DIR, 'plots')
LOGS_FT_OUTPUT_DIR = os.path.join(IMPUTATIONS_FT_BASE_DIR, 'logs')
os.makedirs(METRICS_FT_OUTPUT_DIR, exist_ok=True); os.makedirs(IMPUTED_CSVS_FT_OUTPUT_DIR, exist_ok=True)
os.makedirs(MODELS_FT_OUTPUT_DIR, exist_ok=True); os.makedirs(SCALERS_FT_OUTPUT_DIR, exist_ok=True); os.makedirs(PLOTS_FT_OUTPUT_DIR, exist_ok=True)

# --- Redirección de Log ---
log_file_path = os.path.join(LOGS_FT_OUTPUT_DIR, f'log_{MOMENT_METHOD_NAME_FT}_{"ALL_CITIES" if not TARGET_CITY_FT else sanitize_name(TARGET_CITY_FT)}_{time.strftime("%Y%m%d-%H%M%S")}.txt')
original_stdout, original_stderr = sys.stdout, sys.stderr
class Tee:
    def __init__(self, *files): self.files = files
    def write(self, obj):
        for f in self.files:
            try: f.write(obj); f.flush()
            except (IOError, ValueError): pass
    def flush(self):
        for f in self.files:
            try: f.flush()
            except (IOError, ValueError): pass
log_file = open(log_file_path, 'w', encoding='utf-8')
sys.stdout = Tee(sys.stdout, log_file); sys.stderr = Tee(sys.stderr, log_file)
print(f"--- INICIO DEL SCRIPT: {time.strftime('%Y-%m-%d %H:%M:%S')} ---")

# --- 0. Carga de Datos y Modelo Base ---
print(f"\n--- 0. Carga de Datos y Modelo Base ---")
try:
    file_path_load_orig = os.path.join('CSVs', 'Air_Quality.csv')
    df_air_quality_ground_truth_raw = pd.read_csv(file_path_load_orig)
    print(f"Datos originales cargados desde: {file_path_load_orig}")
except Exception as e:
    print(f"Advertencia: No se pudo cargar el archivo de datos. {e}. Saliendo."); exit()

df_air_quality_ground_truth_raw['Date'] = pd.to_datetime(df_air_quality_ground_truth_raw['Date'], utc=True)
df_air_quality_ground_truth_raw['City'] = df_air_quality_ground_truth_raw['City'].replace('New_York', 'New York')
city_timezones = {'Brasilia': 'America/Sao_Paulo', 'Cairo': 'Africa/Cairo', 'Dubai': 'Asia/Dubai', 'London': 'Europe/London', 'New York': 'America/New_York', 'Sydney': 'Australia/Sydney'}

device = "cuda" if torch.cuda.is_available() else "cpu"; print(f"Usando dispositivo: {device}")
print(f"Cargando modelo MOMENT base: {MODEL_NAME_PRETRAINED_FT}")
moment_base_model = MOMENTPipeline.from_pretrained(MODEL_NAME_PRETRAINED_FT, model_kwargs={'task_name': 'reconstruction'})
moment_base_model.init()
print("Modelo base cargado.")

# --- Funciones de Dataset, Entrenamiento y Evaluación ---

# *** FUNCIÓN DE PLOTEO AÑADIDA ***
def plot_imputation_results_segmented(original_series, imputed_series, nan_mask, variable_name, city_name, scenario_name, output_dir, n_segments=10):
    total_len = len(original_series)
    if total_len == 0: return
        
    segment_len = total_len // n_segments
    scenario_plot_dir = os.path.join(output_dir, scenario_name)
    os.makedirs(scenario_plot_dir, exist_ok=True)

    for i in range(n_segments):
        start = i * segment_len
        end = (i + 1) * segment_len if i < n_segments - 1 else total_len
        if start >= end: continue

        segment_original = original_series.iloc[start:end]
        segment_imputed = imputed_series.iloc[start:end]
        segment_mask = nan_mask.iloc[start:end]
        
        plt.figure(figsize=(20, 8))
        
        plt.plot(segment_original.index, segment_original, color='royalblue', label='Ground Truth', alpha=0.9, zorder=2, linewidth=1.5)
        plt.plot(segment_imputed.index, segment_imputed, color='gray', linestyle='--', label='Observed/Imputed Series', alpha=0.7, zorder=1)
        
        imputed_zone = segment_imputed.where(segment_mask)
        plt.plot(imputed_zone.index, imputed_zone, color='red', linewidth=2.5, label='Imputed Values', zorder=3)
        
        plt.title(f'Imputation: {variable_name} in {city_name} (Segment {i+1}/{n_segments})\nScenario: {scenario_name}')
        plt.xlabel('Time Index'); plt.ylabel(f'{variable_name} Value'); plt.legend()
        plt.grid(True, which='both', linestyle='--', linewidth=0.5); plt.tight_layout()
        
        plot_filename = f"imputation_{scenario_name}_{variable_name}_seg_{i+1}.png"
        plt.savefig(os.path.join(scenario_plot_dir, plot_filename), dpi=150); plt.close()
    
    print(f"    {n_segments} gráficos guardados en: {scenario_plot_dir}")

class MomentFTSimulatedImputationDataset(Dataset):
    def __init__(self, data_array_1d, seq_len, step, missing_ratio, n_replications):
        self.data, self.seq_len, self.step, self.missing_ratio, self.n_replications = data_array_1d.squeeze(), seq_len, step, missing_ratio, n_replications
        self.indices = list(range(0, len(self.data) - self.seq_len + 1, self.step))
        if not self.indices or self.indices[-1] < len(self.data) - self.seq_len: self.indices.append(len(self.data) - self.seq_len)
    def __len__(self): return len(self.indices)
    def __getitem__(self, idx):
        start_idx = self.indices[idx]; seq_1d = self.data[start_idx : start_idx + self.seq_len]
        if len(seq_1d) < self.seq_len: seq_1d = np.pad(seq_1d, (0, self.seq_len - len(seq_1d)), 'constant')
        seq_replicated = np.tile(seq_1d[:, np.newaxis], (1, self.n_replications))
        mask_indices = np.random.choice(self.seq_len, int(self.seq_len * self.missing_ratio), replace=False)
        sim_nan_mask = np.zeros_like(seq_replicated, dtype=bool); sim_nan_mask[mask_indices, :] = True
        input_mask = ~sim_nan_mask.all(axis=1); x_enc = seq_replicated.copy(); x_enc[sim_nan_mask] = 0.0
        return {'x_enc': torch.tensor(x_enc.T, dtype=torch.float32), 'input_mask': torch.tensor(input_mask, dtype=torch.bool), 
                'target_clean': torch.tensor(seq_replicated.T, dtype=torch.float32), 'simulated_nan_mask': torch.tensor(sim_nan_mask.T, dtype=torch.bool)}

class MomentImputationDataset(Dataset):
    def __init__(self, data_array_1d, nan_mask_1d, seq_len, step, n_replications):
        self.data, self.mask, self.seq_len, self.step, self.n_replications = data_array_1d.squeeze(), nan_mask_1d.squeeze(), seq_len, step, n_replications
        self.indices = list(range(0, len(self.data) - self.seq_len + 1, self.step))
        if not self.indices or self.indices[-1] < len(self.data) - self.seq_len: self.indices.append(len(self.data) - self.seq_len)
    def __len__(self): return len(self.indices)
    def __getitem__(self, idx):
        start = self.indices[idx]; seq_1d, mask_1d = self.data[start:start + self.seq_len], self.mask[start:start + self.seq_len]
        if len(seq_1d) < self.seq_len:
            pad_len = self.seq_len - len(seq_1d)
            seq_1d, mask_1d = np.pad(seq_1d, (0, pad_len), 'constant'), np.pad(mask_1d, (0, pad_len), 'constant', constant_values=True)
        seq_replicated = np.tile(seq_1d[:, np.newaxis], (1, self.n_replications))
        return {'timeseries': torch.tensor(seq_replicated.T, dtype=torch.float32), 'input_mask': torch.tensor(~mask_1d, dtype=torch.bool)}

def evaluate_model(model, val_loader, device, criterion):
    model.eval(); total_val_loss, batch_count = 0, 0
    with torch.no_grad():
        for batch_data in val_loader:
            x_enc_batch, input_mask_batch, target_clean_batch, simulated_nan_mask_batch = (batch_data['x_enc'].to(device), batch_data['input_mask'].to(device), batch_data['target_clean'].to(device), batch_data['simulated_nan_mask'].to(device))
            try:
                outputs = model(x_enc=x_enc_batch, input_mask=input_mask_batch); reconstruction_all = outputs.reconstruction
                if simulated_nan_mask_batch.sum() > 0:
                    loss = criterion(reconstruction_all[simulated_nan_mask_batch], target_clean_batch[simulated_nan_mask_batch])
                    if not torch.isnan(loss): total_val_loss += loss.item(); batch_count += 1
            except Exception: continue
    return total_val_loss / batch_count if batch_count > 0 else float('inf')

def train_moment_model(model, train_loader, val_loader, device, epochs, variable_name, model_save_path, learning_rate, lr_patience, es_patience, min_delta_es):
    model.train(); optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate); criterion = nn.MSELoss()
    scheduler = ReduceLROnPlateau(optimizer, 'min', factor=0.5, patience=lr_patience, verbose=False)
    best_val_loss, epochs_no_improve = float('inf'), 0
    for epoch in range(epochs):
        model.train(); total_train_loss, train_batches = 0, 0
        pbar = tqdm(train_loader, desc=f"  Epoch {epoch+1}/{epochs} (Train)", leave=False)
        for batch_data in pbar:
            x_enc, input_mask, target_clean, sim_nan_mask = (batch_data['x_enc'].to(device), batch_data['input_mask'].to(device), batch_data['target_clean'].to(device), batch_data['simulated_nan_mask'].to(device))
            optimizer.zero_grad()
            try:
                outputs = model(x_enc=x_enc, input_mask=input_mask); reconstruction = outputs.reconstruction
                loss = criterion(reconstruction[sim_nan_mask], target_clean[sim_nan_mask]) if sim_nan_mask.sum() > 0 else torch.tensor(0.0)
            except Exception: continue
            if torch.isnan(loss) or not loss.requires_grad: continue
            loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); optimizer.step()
            total_train_loss += loss.item(); train_batches += 1; pbar.set_postfix(loss=loss.item())
        
        avg_train_loss = total_train_loss / train_batches if train_batches > 0 else float('inf')
        avg_val_loss = evaluate_model(model, val_loader, device, criterion)
        print(f"    Epoch {epoch+1}: Train Loss: {avg_train_loss:.6f}, Val Loss: {avg_val_loss:.6f}")
        scheduler.step(avg_val_loss)
        if avg_val_loss < best_val_loss - min_delta_es:
            best_val_loss, epochs_no_improve = avg_val_loss, 0
            torch.save(model.state_dict(), model_save_path)
        else:
            epochs_no_improve += 1
        if epochs_no_improve >= es_patience: print(f"      Early stopping en Epoch {epoch+1}."); break
    print(f"  Fine-tuning completado. Mejor Val Loss: {best_val_loss:.6f}")
    if os.path.exists(model_save_path): model.load_state_dict(torch.load(model_save_path, map_location=device))
    return model

def impute_univar_with_finetuned_model(model_ft_var, df_univar_with_nans_scaled_zeros, original_nan_mask_univar, scaler_var, seq_len_imp, step_imp, batch_size_imp, n_replications_imp, device_str):
    model_ft_var.eval(); imputation_dataset = MomentImputationDataset(df_univar_with_nans_scaled_zeros.values, original_nan_mask_univar.values, seq_len_imp, step_imp, n_replications_imp)
    if not imputation_dataset.indices: return None
    imputation_loader = DataLoader(imputation_dataset, batch_size=batch_size_imp, shuffle=False, num_workers=0)
    n_rows_total, n_features_total = df_univar_with_nans_scaled_zeros.shape
    accumulated_preds, accumulated_counts = np.zeros((n_rows_total, n_features_total)), np.zeros((n_rows_total, n_features_total))
    with torch.no_grad():
        for batch_idx, batch_data in enumerate(tqdm(imputation_loader, desc=f"  Imputando", leave=False, unit="batch")):
            timeseries_batch, input_mask_batch = batch_data['timeseries'].to(device_str), batch_data['input_mask'].to(device_str)
            try: outputs = model_ft_var(x_enc=timeseries_batch, input_mask=input_mask_batch); preds_batch = outputs.reconstruction.mean(dim=1).cpu().numpy()
            except Exception: continue
            for i in range(preds_batch.shape[0]):
                start_idx = imputation_dataset.indices[batch_idx * batch_size_imp + i]; end_idx = min(start_idx + seq_len_imp, n_rows_total)
                len_add = min(preds_batch.shape[1], end_idx - start_idx)
                accumulated_preds[start_idx:end_idx] += preds_batch[i, :len_add].reshape(-1, 1); accumulated_counts[start_idx:end_idx] += 1
    accumulated_counts[accumulated_counts == 0] = 1; imputed_scaled = accumulated_preds / accumulated_counts
    imputed_unscaled = scaler_var.inverse_transform(imputed_scaled)
    df_final = df_univar_with_nans_scaled_zeros.copy(); df_final.iloc[:, 0] = scaler_var.inverse_transform(df_final.values)
    mask_1d = original_nan_mask_univar.iloc[:, 0]
    df_final.loc[mask_1d] = imputed_unscaled[mask_1d]
    return df_final

def calculate_metrics_moment(y_true_df, y_pred_df, nan_mask_df, cols_to_eval):
    results = {}; y_true, y_pred, mask = y_true_df.reset_index(drop=True), y_pred_df.reset_index(drop=True), nan_mask_df.reset_index(drop=True)
    for col in cols_to_eval:
        if col not in y_true.columns or col not in y_pred.columns: continue
        results[col] = {'MAE': np.nan, 'RMSE': np.nan, 'NRMAE_range': np.nan, 'NRMSE_range': np.nan}
        true_vals, pred_vals, m = y_true[col], y_pred[col], mask[col].astype(bool)
        if m.sum() == 0: continue
        mae = mean_absolute_error(true_vals[m], pred_vals[m]); rmse = np.sqrt(mean_squared_error(true_vals[m], pred_vals[m]))
        data_range = y_true[col].max() - y_true[col].min()
        results[col].update({'MAE': mae, 'RMSE': rmse, 'NRMAE_range': mae / data_range if data_range > 1e-9 else np.nan, 'NRMSE_range': rmse / data_range if data_range > 1e-9 else np.nan})
    return results

# --- Script Principal ---
all_detailed_metrics_ft_results = []
script_start_time_ft = time.time()
cities_to_process = [TARGET_CITY_FT] if TARGET_CITY_FT else df_air_quality_ground_truth_raw['City'].unique()

for city in cities_to_process:
    print(f"\n\n==================== PROCESANDO CIUDAD: {city} ====================")
    city_name_safe = sanitize_name(city)
    
    df_city_clean = df_air_quality_ground_truth_raw[df_air_quality_ground_truth_raw['City'] == city].copy()
    if df_city_clean.empty: continue
    
    if city in city_timezones: df_city_clean['Date'] = df_city_clean['Date'].dt.tz_convert(city_timezones[city])
    
    for col in ALL_TARGET_COLS_FT:
        if col in df_city_clean.columns: df_city_clean[col] = df_city_clean[col].ffill().bfill()
    if df_city_clean[ALL_TARGET_COLS_FT].isnull().any().any(): continue
    
    city_models_dir = os.path.join(MODELS_FT_OUTPUT_DIR, city_name_safe); os.makedirs(city_models_dir, exist_ok=True)
    city_scalers_dir = os.path.join(SCALERS_FT_OUTPUT_DIR, city_name_safe); os.makedirs(city_scalers_dir, exist_ok=True)
    city_plots_dir = os.path.join(PLOTS_FT_OUTPUT_DIR, city_name_safe); os.makedirs(city_plots_dir, exist_ok=True)
    
    finetuned_models_dict, scalers_dict = {}, {}
    
    for var_to_finetune in ALL_TARGET_COLS_FT:
        print(f"\n--- Fine-Tuning para: {var_to_finetune} en {city} ---")
        
        scaler = MinMaxScaler(feature_range=(-1, 1))
        data_to_scale = df_city_clean[[var_to_finetune]].values
        if len(data_to_scale) < 1: continue
        scaler.fit(data_to_scale); scalers_dict[var_to_finetune] = scaler
        joblib.dump(scaler, os.path.join(city_scalers_dir, f'scaler_{sanitize_name(var_to_finetune)}.joblib'))
        
        scaled_data = scaler.transform(data_to_scale)
        split_idx = int(len(scaled_data) * FINETUNE_TRAIN_RATIO_NEW)
        train_data, val_data = scaled_data[:split_idx], scaled_data[split_idx:]

        if len(train_data) < SEQ_LEN_MODEL_FT or len(val_data) < SEQ_LEN_MODEL_FT: continue

        train_dataset = MomentFTSimulatedImputationDataset(train_data, SEQ_LEN_MODEL_FT, FINETUNE_STEP_DATALOADER_NEW, SIMULATED_MISSING_RATIO_FOR_FT, EXPECTED_MODEL_CHANNELS)
        train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE_FINETUNE_NEW, shuffle=True, num_workers=0, drop_last=True)
        val_dataset = MomentFTSimulatedImputationDataset(val_data, SEQ_LEN_MODEL_FT, FINETUNE_STEP_DATALOADER_NEW, SIMULATED_MISSING_RATIO_FOR_FT, EXPECTED_MODEL_CHANNELS)
        val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE_FINETUNE_NEW, shuffle=False)
        
        if len(train_loader) == 0 or len(val_loader) == 0: continue
            
        model = MOMENTPipeline.from_pretrained(MODEL_NAME_PRETRAINED_FT, model_kwargs={'task_name': 'reconstruction'})
        model.init(); model.load_state_dict(moment_base_model.state_dict()); model = model.to(device)
        model_save_path = os.path.join(city_models_dir, f'moment_ft_best_{sanitize_name(var_to_finetune)}.pth')
        
        finetuned_model = train_moment_model(model, train_loader, val_loader, device, FINETUNE_EPOCHS, var_to_finetune, model_save_path, LEARNING_RATE_FINETUNE_NEW, LR_PATIENCE_NEW, PATIENCE_EARLY_STOPPING_NEW, MIN_DELTA_EARLY_STOPPING_NEW)
        finetuned_models_dict[var_to_finetune] = finetuned_model.to('cpu') if finetuned_model else None
        torch.cuda.empty_cache()

    print(f"\n--- Iniciando Fase de Imputación para {city} ---")
    df_city_ground_truth_ft_reset = df_city_clean.reset_index(drop=True)
    city_imputed_csv_dir = os.path.join(IMPUTED_CSVS_FT_OUTPUT_DIR, city_name_safe); os.makedirs(city_imputed_csv_dir, exist_ok=True)
    
    for scenario in NAN_SIMULATION_SCENARIOS_FT:
        scenario_name_for_dir = scenario['name']
        scenario_name_for_file = scenario['name'].replace('%', 'percent')
        print(f"\nProcesando Escenario: {scenario_name_for_dir}")
        sim_data_dir = os.path.join(SIMULATED_DATA_INPUT_ROOT_DIR_FT, city_name_safe, scenario_name_for_dir)
        imputed_scenario_dir = os.path.join(city_imputed_csv_dir, scenario_name_for_dir); os.makedirs(imputed_scenario_dir, exist_ok=True)
        if not os.path.exists(sim_data_dir): continue
        
        for rep in range(1, N_REPETITIONS_FT_EVAL + 1):
            sim_files = glob.glob(os.path.join(sim_data_dir, f"{city_name_safe}_{scenario_name_for_file}_rep{rep}_*_with_nans.csv"))
            if not sim_files: continue
            
            sim_file_path = sim_files[0]
            seed = os.path.basename(sim_file_path).split('_seed')[-1].split('_with_nans.csv')[0]
            df_with_nans = pd.read_csv(sim_file_path, parse_dates=['Date'])
            df_imputed = df_with_nans.copy()
            
            for var in tqdm(ALL_TARGET_COLS_FT, desc=f"  Imputando Rep {rep}", unit="var"):
                model, scaler = finetuned_models_dict.get(var), scalers_dict.get(var)
                if not model or not scaler: continue
                model = model.to('cpu')
                
                univar_with_nans = df_with_nans[[var]].copy()
                nan_mask = univar_with_nans.isnull()
                scaled_zeros = pd.DataFrame(scaler.transform(univar_with_nans.fillna(0)), columns=[var])
                
                imputed_univar = impute_univar_with_finetuned_model(model, scaled_zeros, nan_mask, scaler, SEQ_LEN_MODEL_FT, IMPUTATION_STEP_FT, BATCH_SIZE_IMPUTE_FT, EXPECTED_MODEL_CHANNELS, 'cpu')
                if imputed_univar is not None: df_imputed[var] = imputed_univar[var]

            imputed_filename = os.path.basename(sim_file_path).replace("_with_nans.csv", "_imputed.csv")
            df_imputed.to_csv(os.path.join(imputed_scenario_dir, imputed_filename), index=False)
            
            nan_mask_for_metrics = df_with_nans[ALL_TARGET_COLS_FT].isnull() & ~df_city_ground_truth_ft_reset[ALL_TARGET_COLS_FT].isnull()
            metrics = calculate_metrics_moment(df_city_ground_truth_ft_reset, df_imputed, nan_mask_for_metrics, ALL_TARGET_COLS_FT)
            for var_name, scores in metrics.items():
                all_detailed_metrics_ft_results.append({'City': city, 'Scenario': scenario_name_for_dir, 'Repetition': rep, 'Seed': seed, 'Imputation_Method': MOMENT_METHOD_NAME_FT, 'Variable': var_name, **scores})

            if rep == 1: # Plot only for the first repetition
                plot_imputation_results_segmented(
                    original_series=df_city_ground_truth_ft_reset['AQI'],
                    imputed_series=df_imputed['AQI'],
                    nan_mask=nan_mask_for_metrics['AQI'],
                    variable_name='AQI', city_name=city, scenario_name=scenario_name_for_dir,
                    output_dir=city_plots_dir
                )

if all_detailed_metrics_ft_results:
    df_metrics = pd.DataFrame(all_detailed_metrics_ft_results)
    sufijo_ft = f"_{sanitize_name(TARGET_CITY_FT)}" if TARGET_CITY_FT else "_ALL_CITIES"
    global_detailed_ft_file = os.path.join(METRICS_FT_OUTPUT_DIR, f'ALL_metrics_detailed_{MOMENT_METHOD_NAME_FT}{sufijo_ft}.csv')
    df_metrics.to_csv(global_detailed_ft_file, index=False)
    print(f"\nMétricas detalladas globales guardadas en: {global_detailed_ft_file}")
    
    print("\nGenerando resúmenes de métricas por variable y ciudad...")
    grouped_by_city_var_ft = df_metrics.groupby(['City', 'Variable'])[['MAE', 'RMSE', 'NRMAE_range', 'NRMSE_range']].mean().reset_index()
    grouped_by_city_var_ft.rename(columns={'MAE': 'Avg_MAE', 'RMSE': 'Avg_RMSE', 'NRMAE_range': 'Avg_NRMAE_range', 'NRMSE_range': 'Avg_NRMSE_range'}, inplace=True)
    for city_name_summary in grouped_by_city_var_ft['City'].unique():
        city_name_safe_summary = sanitize_name(city_name_summary)
        city_metrics_df = grouped_by_city_var_ft[grouped_by_city_var_ft['City'] == city_name_summary]
        variables_summary_path = os.path.join(METRICS_FT_OUTPUT_DIR, f'metrics_variables_summary_{city_name_safe_summary}.csv')
        city_metrics_df.to_csv(variables_summary_path, index=False)
        print(f"  Resumen de variables para {city_name_summary} guardado en: {variables_summary_path}")

    print("\nGenerando resumen combinado de contaminantes...")
    df_contaminants_only_ft = df_metrics[df_metrics['Variable'].isin(CONTAMINANT_COLS_FT)]
    if not df_contaminants_only_ft.empty:
        contaminants_combined_summary_ft = df_contaminants_only_ft.groupby('City')[['MAE', 'RMSE', 'NRMAE_range', 'NRMSE_range']].mean().reset_index()
        contaminants_combined_summary_ft.rename(columns={'MAE': 'Avg_MAE_Conts', 'RMSE': 'Avg_RMSE_Conts', 'NRMAE_range': 'Avg_NRMAE_range_Conts', 'NRMSE_range': 'Avg_NRMSE_range_Conts'}, inplace=True)
        contaminants_summary_path = os.path.join(METRICS_FT_OUTPUT_DIR, 'metrics_contaminants_combined_summary.csv')
        contaminants_combined_summary_ft.to_csv(contaminants_summary_path, index=False)
        print(f"  Resumen contaminantes guardado: {contaminants_summary_path}")

    print("\nGenerando resumen combinado de AQI...")
    df_aqi_only_ft = df_metrics[df_metrics['Variable'] == 'AQI']
    if not df_aqi_only_ft.empty:
        aqi_combined_summary_ft = df_aqi_only_ft.groupby('City')[['MAE', 'RMSE', 'NRMAE_range', 'NRMSE_range']].mean().reset_index()
        aqi_combined_summary_ft.rename(columns={'MAE': 'Avg_MAE_AQI', 'RMSE': 'Avg_RMSE_AQI', 'NRMAE_range': 'Avg_NRMAE_range_AQI', 'NRMSE_range': 'Avg_NRMSE_range_AQI'}, inplace=True)
        aqi_summary_path = os.path.join(METRICS_FT_OUTPUT_DIR, 'metrics_aqi_combined_summary.csv')
        aqi_combined_summary_ft.to_csv(aqi_summary_path, index=False)
        print(f"  Resumen AQI guardado: {aqi_summary_path}")

print(f"\nProceso completado. Tiempo total: {(time.time() - script_start_time_ft) / 60:.2f} minutos.")
if 'log_file' in locals() and log_file and not log_file.closed:
    sys.stdout = original_stdout; sys.stderr = original_stderr;
    log_file.close()
    print(f"Salida de log completada y guardada en: {log_file_path}")

FileNotFoundError: [Errno 2] No such file or directory: 'c:\\Users\\felix\\Documents\\MÁSTER\\TFM\\TFM_Ambiental\\AIR QUALITY\\imputaciones\\MOMENT_FT_final_MOMENT_1_large\\logs\\log_MOMENT_FT_final_MOMENT_1_large_ALL_CITIES_20250623-203306.txt'

In [ ]:
import pandas as pd
import numpy as np
import os
import torch
from torch.utils.data import DataLoader, Dataset
import torch.nn as nn
from torch.optim.lr_scheduler import ReduceLROnPlateau
from tqdm import tqdm
import joblib
import traceback
import time
import random
import warnings
import shutil
import glob
import sys
import matplotlib.pyplot as plt
import seaborn as sns

# Attempt to import MOMENT, handle if not found
try:
    from momentfm import MOMENTPipeline
except ImportError:
    print("MOMENTfm library not found. Please install it: pip install momentfm")
    exit()

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

warnings.filterwarnings("ignore")

# --- Configuración de Estilo para Gráficas ---
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (20, 8)
plt.rcParams['axes.titlesize'] = 18
plt.rcParams['axes.labelsize'] = 14
plt.rcParams['xtick.labelsize'] = 12
plt.rcParams['ytick.labelsize'] = 12
plt.rcParams['legend.fontsize'] = 14

# --- Helper Function for Sanitizing File/Path Names ---
def sanitize_name(name_str):
    import re
    s = str(name_str)
    s = re.sub(r'[^\w\s-]', '', s).strip()
    s = re.sub(r'[-\s]+', '_', s)
    s = re.sub(r'_+', '_', s)
    return s if s else "default_sanitized_name"

# --- Parámetros Configurables ---
TARGET_CITY_FT = "Brasilia"  # Cambiar a None para procesar todas las ciudades
CONTAMINANT_COLS_FT = ['CO', 'NO2', 'SO2', 'O3', 'PM2.5', 'PM10']
AQI_COL_FT = ['AQI']
ALL_TARGET_COLS_FT = CONTAMINANT_COLS_FT + AQI_COL_FT
SEQ_LEN_MODEL_FT = 512
IMPUTATION_STEP_FT = SEQ_LEN_MODEL_FT
BATCH_SIZE_IMPUTE_FT = 128
MODEL_NAME_PRETRAINED_FT = "AutonLab/MOMENT-1-large"
EXPECTED_MODEL_CHANNELS = 8

# Fine-tuning Parameters
FINETUNE_EPOCHS = 30
FINETUNE_TRAIN_RATIO_NEW = 0.8
FINETUNE_STEP_DATALOADER_NEW = 128
BATCH_SIZE_FINETUNE_NEW = 64
LEARNING_RATE_FINETUNE_NEW = 1e-4
SIMULATED_MISSING_RATIO_FOR_FT = 0.25
PATIENCE_EARLY_STOPPING_NEW = 20
LR_PATIENCE_NEW = 8
MIN_DELTA_EARLY_STOPPING_NEW = 1e-5

N_REPETITIONS_FT_EVAL = 1
NAN_SIMULATION_SCENARIOS_FT = [{'name': name} for name in ['RandomRows_20percent', 'BlockRows_12h_20percent', 'BlockRows_24h_20percent', 'BlockRows_36h_20percent', 'BlockRows_48h_20percent', 'BlockRows_60h_20percent']]

# --- Rutas ---
PROJECT_ROOT_DIR_FT = os.getcwd()
SIMULATED_DATA_INPUT_ROOT_DIR_FT = os.path.join(PROJECT_ROOT_DIR_FT, 'CSVs', 'simulados')
MOMENT_METHOD_NAME_FT = f'MOMENT_FT_full_v2_{sanitize_name(MODEL_NAME_PRETRAINED_FT.split("/")[-1])}'
IMPUTATIONS_FT_BASE_DIR = os.path.join(PROJECT_ROOT_DIR_FT, 'imputaciones', MOMENT_METHOD_NAME_FT)
METRICS_FT_OUTPUT_DIR = os.path.join(IMPUTATIONS_FT_BASE_DIR, 'metricas')
IMPUTED_CSVS_FT_OUTPUT_DIR = os.path.join(IMPUTATIONS_FT_BASE_DIR, 'CSVs')
MODELS_FT_OUTPUT_DIR = os.path.join(IMPUTATIONS_FT_BASE_DIR, 'modelos_ft_sim_impute')
SCALERS_FT_OUTPUT_DIR = os.path.join(IMPUTATIONS_FT_BASE_DIR, 'scalers')
PLOTS_FT_OUTPUT_DIR = os.path.join(IMPUTATIONS_FT_BASE_DIR, 'plots')
LOGS_FT_OUTPUT_DIR = os.path.join(IMPUTATIONS_FT_BASE_DIR, 'logs')
os.makedirs(METRICS_FT_OUTPUT_DIR, exist_ok=True); os.makedirs(IMPUTED_CSVS_FT_OUTPUT_DIR, exist_ok=True)
os.makedirs(MODELS_FT_OUTPUT_DIR, exist_ok=True); os.makedirs(SCALERS_FT_OUTPUT_DIR, exist_ok=True); os.makedirs(LOGS_FT_OUTPUT_DIR, exist_ok=True)

# --- Redirección de Log ---
# CORRECCIÓN: Eliminar los dos puntos (:) del formato de fecha y hora para compatibilidad con Windows
log_file_path = os.path.join(LOGS_FT_OUTPUT_DIR, f'log_{MOMENT_METHOD_NAME_FT}_{"ALL_CITIES" if not TARGET_CITY_FT else sanitize_name(TARGET_CITY_FT)}_{time.strftime("%Y%m%d-%H%M%S")}.txt')
original_stdout, original_stderr = sys.stdout, sys.stderr
class Tee:
    def __init__(self, *files): self.files = files
    def write(self, obj):
        for f in self.files:
            try: f.write(obj); f.flush()
            except (IOError, ValueError): pass
    def flush(self):
        for f in self.files:
            try: f.flush()
            except (IOError, ValueError): pass
log_file = open(log_file_path, 'w', encoding='utf-8')
sys.stdout = Tee(sys.stdout, log_file); sys.stderr = Tee(sys.stderr, log_file)
print(f"--- INICIO DEL SCRIPT: {time.strftime('%Y-%m-%d %H:%M:%S')} ---")

# --- 0. Carga de Datos y Modelo Base ---
print(f"\n--- 0. Carga de Datos y Modelo Base ---")
try:
    file_path_load_orig = os.path.join('CSVs', 'Air_Quality.csv')
    df_air_quality_ground_truth_raw = pd.read_csv(file_path_load_orig)
    print(f"Datos originales cargados desde: {file_path_load_orig}")
except Exception as e:
    print(f"Advertencia: No se pudo cargar el archivo de datos. {e}. Saliendo."); exit()

df_air_quality_ground_truth_raw['Date'] = pd.to_datetime(df_air_quality_ground_truth_raw['Date'], utc=True)
df_air_quality_ground_truth_raw['City'] = df_air_quality_ground_truth_raw['City'].replace('New_York', 'New York')
city_timezones = {'Brasilia': 'America/Sao_Paulo', 'Cairo': 'Africa/Cairo', 'Dubai': 'Asia/Dubai', 'London': 'Europe/London', 'New York': 'America/New_York', 'Sydney': 'Australia/Sydney'}

device = "cuda" if torch.cuda.is_available() else "cpu"; print(f"Usando dispositivo: {device}")
print(f"Cargando modelo MOMENT base: {MODEL_NAME_PRETRAINED_FT}")
moment_base_model = MOMENTPipeline.from_pretrained(MODEL_NAME_PRETRAINED_FT, model_kwargs={'task_name': 'reconstruction'})
moment_base_model.init()
print("Modelo base cargado.")

# --- Funciones de Dataset, Entrenamiento y Evaluación ---
def plot_imputation_results_segmented(original_series, imputed_series, nan_mask, variable_name, city_name, scenario_name, output_dir, n_segments=10):
    total_len = len(original_series)
    if total_len == 0: return
    segment_len = total_len // n_segments
    scenario_plot_dir = os.path.join(output_dir, scenario_name)
    os.makedirs(scenario_plot_dir, exist_ok=True)
    for i in range(n_segments):
        start = i * segment_len
        end = (i + 1) * segment_len if i < n_segments - 1 else total_len
        if start >= end: continue
        segment_original = original_series.iloc[start:end]
        segment_imputed = imputed_series.iloc[start:end]
        segment_mask = nan_mask.iloc[start:end]
        plt.figure(figsize=(20, 8))
        plt.plot(segment_original.index, segment_original, color='royalblue', label='Ground Truth', alpha=0.9, zorder=2, linewidth=1.5)
        plt.plot(segment_imputed.index, segment_imputed, color='gray', linestyle='--', label='Observed/Imputed Series', alpha=0.7, zorder=1)
        imputed_zone = segment_imputed.where(segment_mask)
        plt.plot(imputed_zone.index, imputed_zone, color='red', linewidth=2.5, label='Imputed Values', zorder=3)
        plt.title(f'Imputation: {variable_name} in {city_name} (Segment {i+1}/{n_segments})\nScenario: {scenario_name}')
        plt.xlabel('Time Index'); plt.ylabel(f'{variable_name} Value'); plt.legend()
        plt.grid(True, which='both', linestyle='--', linewidth=0.5); plt.tight_layout()
        plot_filename = f"imputation_{scenario_name}_{variable_name}_seg_{i+1}.png"
        plt.savefig(os.path.join(scenario_plot_dir, plot_filename), dpi=150); plt.close()
    print(f"    {n_segments} gráficos guardados en: {scenario_plot_dir}")

class MomentFTSimulatedImputationDataset(Dataset):
    def __init__(self, data_array_1d, seq_len, step, missing_ratio, n_replications):
        self.data, self.seq_len, self.step, self.missing_ratio, self.n_replications = data_array_1d.squeeze(), seq_len, step, missing_ratio, n_replications
        self.indices = list(range(0, len(self.data) - self.seq_len + 1, self.step))
        if not self.indices or self.indices[-1] < len(self.data) - self.seq_len: self.indices.append(len(self.data) - self.seq_len)
    def __len__(self): return len(self.indices)
    def __getitem__(self, idx):
        start_idx = self.indices[idx]; seq_1d = self.data[start_idx : start_idx + self.seq_len]
        if len(seq_1d) < self.seq_len: seq_1d = np.pad(seq_1d, (0, self.seq_len - len(seq_1d)), 'constant')
        seq_replicated = np.tile(seq_1d[:, np.newaxis], (1, self.n_replications))
        mask_indices = np.random.choice(self.seq_len, int(self.seq_len * self.missing_ratio), replace=False)
        sim_nan_mask = np.zeros_like(seq_replicated, dtype=bool); sim_nan_mask[mask_indices, :] = True
        input_mask = ~sim_nan_mask.all(axis=1); x_enc = seq_replicated.copy(); x_enc[sim_nan_mask] = 0.0
        return {'x_enc': torch.tensor(x_enc.T, dtype=torch.float32), 'input_mask': torch.tensor(input_mask, dtype=torch.bool), 
                'target_clean': torch.tensor(seq_replicated.T, dtype=torch.float32), 'simulated_nan_mask': torch.tensor(sim_nan_mask.T, dtype=torch.bool)}

class MomentImputationDataset(Dataset):
    def __init__(self, data_array_1d, nan_mask_1d, seq_len, step, n_replications):
        self.data, self.mask, self.seq_len, self.step, self.n_replications = data_array_1d.squeeze(), nan_mask_1d.squeeze(), seq_len, step, n_replications
        self.indices = list(range(0, len(self.data) - self.seq_len + 1, self.step))
        if not self.indices or self.indices[-1] < len(self.data) - self.seq_len: self.indices.append(len(self.data) - self.seq_len)
    def __len__(self): return len(self.indices)
    def __getitem__(self, idx):
        start = self.indices[idx]; seq_1d, mask_1d = self.data[start:start + self.seq_len], self.mask[start:start + self.seq_len]
        if len(seq_1d) < self.seq_len:
            pad_len = self.seq_len - len(seq_1d)
            seq_1d, mask_1d = np.pad(seq_1d, (0, pad_len), 'constant'), np.pad(mask_1d, (0, pad_len), 'constant', constant_values=True)
        seq_replicated = np.tile(seq_1d[:, np.newaxis], (1, self.n_replications))
        return {'timeseries': torch.tensor(seq_replicated.T, dtype=torch.float32), 'input_mask': torch.tensor(~mask_1d, dtype=torch.bool)}

def evaluate_model(model, val_loader, device, criterion):
    model.eval(); total_val_loss, batch_count = 0, 0
    with torch.no_grad():
        for batch_data in val_loader:
            x_enc_batch, input_mask_batch, target_clean_batch, simulated_nan_mask_batch = (batch_data['x_enc'].to(device), batch_data['input_mask'].to(device), batch_data['target_clean'].to(device), batch_data['simulated_nan_mask'].to(device))
            try:
                outputs = model(x_enc=x_enc_batch, input_mask=input_mask_batch); reconstruction_all = outputs.reconstruction
                if simulated_nan_mask_batch.sum() > 0:
                    loss = criterion(reconstruction_all[simulated_nan_mask_batch], target_clean_batch[simulated_nan_mask_batch])
                    if not torch.isnan(loss): total_val_loss += loss.item(); batch_count += 1
            except Exception: continue
    return total_val_loss / batch_count if batch_count > 0 else float('inf')

def train_moment_model(model, train_loader, val_loader, device, epochs, variable_name, model_save_path, learning_rate, lr_patience, es_patience, min_delta_es):
    model.train(); optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate); criterion = nn.MSELoss()
    scheduler = ReduceLROnPlateau(optimizer, 'min', factor=0.5, patience=lr_patience, verbose=False)
    best_val_loss, epochs_no_improve = float('inf'), 0
    for epoch in range(epochs):
        model.train(); total_train_loss, train_batches = 0, 0
        pbar = tqdm(train_loader, desc=f"  Epoch {epoch+1}/{epochs} (Train)", leave=False)
        for batch_data in pbar:
            x_enc, input_mask, target_clean, sim_nan_mask = (batch_data['x_enc'].to(device), batch_data['input_mask'].to(device), batch_data['target_clean'].to(device), batch_data['simulated_nan_mask'].to(device))
            optimizer.zero_grad()
            try:
                outputs = model(x_enc=x_enc, input_mask=input_mask); reconstruction = outputs.reconstruction
                loss = criterion(reconstruction[sim_nan_mask], target_clean[sim_nan_mask]) if sim_nan_mask.sum() > 0 else torch.tensor(0.0)
            except Exception: continue
            if torch.isnan(loss) or not loss.requires_grad: continue
            loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); optimizer.step()
            total_train_loss += loss.item(); train_batches += 1; pbar.set_postfix(loss=loss.item())
        
        avg_train_loss = total_train_loss / train_batches if train_batches > 0 else float('inf')
        avg_val_loss = evaluate_model(model, val_loader, device, criterion)
        print(f"    Epoch {epoch+1}: Train Loss: {avg_train_loss:.6f}, Val Loss: {avg_val_loss:.6f}")
        scheduler.step(avg_val_loss)
        if avg_val_loss < best_val_loss - min_delta_es:
            best_val_loss, epochs_no_improve = avg_val_loss, 0
            torch.save(model.state_dict(), model_save_path)
        else:
            epochs_no_improve += 1
        if epochs_no_improve >= es_patience: print(f"      Early stopping en Epoch {epoch+1}."); break
    print(f"  Fine-tuning completado. Mejor Val Loss: {best_val_loss:.6f}")
    if os.path.exists(model_save_path): model.load_state_dict(torch.load(model_save_path, map_location=device))
    return model

def impute_univar_with_finetuned_model(model_ft_var, df_univar_with_nans_scaled_zeros, original_nan_mask_univar, scaler_var, seq_len_imp, step_imp, batch_size_imp, n_replications_imp, device_str):
    model_ft_var.eval(); imputation_dataset = MomentImputationDataset(df_univar_with_nans_scaled_zeros.values, original_nan_mask_univar.values, seq_len_imp, step_imp, n_replications_imp)
    if not imputation_dataset.indices: return None
    imputation_loader = DataLoader(imputation_dataset, batch_size=batch_size_imp, shuffle=False, num_workers=0)
    n_rows_total, n_features_total = df_univar_with_nans_scaled_zeros.shape
    accumulated_preds, accumulated_counts = np.zeros((n_rows_total, n_features_total)), np.zeros((n_rows_total, n_features_total))
    with torch.no_grad():
        for batch_idx, batch_data in enumerate(tqdm(imputation_loader, desc=f"  Imputando", leave=False, unit="batch")):
            timeseries_batch, input_mask_batch = batch_data['timeseries'].to(device_str), batch_data['input_mask'].to(device_str)
            try: outputs = model_ft_var(x_enc=timeseries_batch, input_mask=input_mask_batch); preds_batch = outputs.reconstruction.mean(dim=1).cpu().numpy()
            except Exception: continue
            for i in range(preds_batch.shape[0]):
                start_idx = imputation_dataset.indices[batch_idx * batch_size_imp + i]; end_idx = min(start_idx + seq_len_imp, n_rows_total)
                len_add = min(preds_batch.shape[1], end_idx - start_idx)
                accumulated_preds[start_idx:end_idx] += preds_batch[i, :len_add].reshape(-1, 1); accumulated_counts[start_idx:end_idx] += 1
    accumulated_counts[accumulated_counts == 0] = 1; imputed_scaled = accumulated_preds / accumulated_counts
    imputed_unscaled = scaler_var.inverse_transform(imputed_scaled)
    df_final = df_univar_with_nans_scaled_zeros.copy(); df_final.iloc[:, 0] = scaler_var.inverse_transform(df_final.values)
    mask_1d = original_nan_mask_univar.iloc[:, 0]
    df_final.loc[mask_1d] = imputed_unscaled[mask_1d]
    return df_final

def calculate_metrics_moment(y_true_df, y_pred_df, nan_mask_df, cols_to_eval):
    results = {}; y_true, y_pred, mask = y_true_df.reset_index(drop=True), y_pred_df.reset_index(drop=True), nan_mask_df.reset_index(drop=True)
    for col in cols_to_eval:
        if col not in y_true.columns or col not in y_pred.columns: continue
        results[col] = {'MAE': np.nan, 'RMSE': np.nan, 'NRMAE_range': np.nan, 'NRMSE_range': np.nan}
        true_vals, pred_vals, m = y_true[col], y_pred[col], mask[col].astype(bool)
        if m.sum() == 0: continue
        mae = mean_absolute_error(true_vals[m], pred_vals[m]); rmse = np.sqrt(mean_squared_error(true_vals[m], pred_vals[m]))
        data_range = y_true[col].max() - y_true[col].min()
        results[col].update({'MAE': mae, 'RMSE': rmse, 'NRMAE_range': mae / data_range if data_range > 1e-9 else np.nan, 'NRMSE_range': rmse / data_range if data_range > 1e-9 else np.nan})
    return results

# --- Script Principal ---
all_detailed_metrics_ft_results = []
script_start_time_ft = time.time()
cities_to_process = [TARGET_CITY_FT] if TARGET_CITY_FT else df_air_quality_ground_truth_raw['City'].unique()

for city in cities_to_process:
    print(f"\n\n==================== PROCESANDO CIUDAD: {city} ====================")
    city_name_safe = sanitize_name(city)
    
    df_city_clean = df_air_quality_ground_truth_raw[df_air_quality_ground_truth_raw['City'] == city].copy()
    if df_city_clean.empty: continue
    
    if city in city_timezones: df_city_clean['Date'] = df_city_clean['Date'].dt.tz_convert(city_timezones[city])
    
    for col in ALL_TARGET_COLS_FT:
        if col in df_city_clean.columns: df_city_clean[col] = df_city_clean[col].ffill().bfill()
    if df_city_clean[ALL_TARGET_COLS_FT].isnull().any().any(): continue
    
    city_models_dir = os.path.join(MODELS_FT_OUTPUT_DIR, city_name_safe); os.makedirs(city_models_dir, exist_ok=True)
    city_scalers_dir = os.path.join(SCALERS_FT_OUTPUT_DIR, city_name_safe); os.makedirs(city_scalers_dir, exist_ok=True)
    city_plots_dir = os.path.join(PLOTS_FT_OUTPUT_DIR, city_name_safe); os.makedirs(city_plots_dir, exist_ok=True)
    
    finetuned_models_dict, scalers_dict = {}, {}
    
    for var_to_finetune in ALL_TARGET_COLS_FT:
        print(f"\n--- Fine-Tuning para: {var_to_finetune} en {city} ---")
        
        scaler = MinMaxScaler(feature_range=(-1, 1))
        data_to_scale = df_city_clean[[var_to_finetune]].values
        if len(data_to_scale) < 1: continue
        scaler.fit(data_to_scale); scalers_dict[var_to_finetune] = scaler
        joblib.dump(scaler, os.path.join(city_scalers_dir, f'scaler_{sanitize_name(var_to_finetune)}.joblib'))
        
        scaled_data = scaler.transform(data_to_scale)
        split_idx = int(len(scaled_data) * FINETUNE_TRAIN_RATIO_NEW)
        train_data, val_data = scaled_data[:split_idx], scaled_data[split_idx:]

        if len(train_data) < SEQ_LEN_MODEL_FT or len(val_data) < SEQ_LEN_MODEL_FT: continue

        train_dataset = MomentFTSimulatedImputationDataset(train_data, SEQ_LEN_MODEL_FT, FINETUNE_STEP_DATALOADER_NEW, SIMULATED_MISSING_RATIO_FOR_FT, EXPECTED_MODEL_CHANNELS)
        train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE_FINETUNE_NEW, shuffle=True, num_workers=0, drop_last=True)
        val_dataset = MomentFTSimulatedImputationDataset(val_data, SEQ_LEN_MODEL_FT, FINETUNE_STEP_DATALOADER_NEW, SIMULATED_MISSING_RATIO_FOR_FT, EXPECTED_MODEL_CHANNELS)
        val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE_FINETUNE_NEW, shuffle=False)
        
        if len(train_loader) == 0 or len(val_loader) == 0: continue
            
        model = MOMENTPipeline.from_pretrained(MODEL_NAME_PRETRAINED_FT, model_kwargs={'task_name': 'reconstruction'})
        model.init(); model.load_state_dict(moment_base_model.state_dict()); model = model.to(device)
        model_save_path = os.path.join(city_models_dir, f'moment_ft_best_{sanitize_name(var_to_finetune)}.pth')
        
        finetuned_model = train_moment_model(model, train_loader, val_loader, device, FINETUNE_EPOCHS, var_to_finetune, model_save_path, LEARNING_RATE_FINETUNE_NEW, LR_PATIENCE_NEW, PATIENCE_EARLY_STOPPING_NEW, MIN_DELTA_EARLY_STOPPING_NEW)
        finetuned_models_dict[var_to_finetune] = finetuned_model.to('cpu') if finetuned_model else None
        torch.cuda.empty_cache()

    print(f"\n--- Iniciando Fase de Imputación para {city} ---")
    df_city_ground_truth_ft_reset = df_city_clean.reset_index(drop=True)
    city_imputed_csv_dir = os.path.join(IMPUTED_CSVS_FT_OUTPUT_DIR, city_name_safe); os.makedirs(city_imputed_csv_dir, exist_ok=True)
    
    for scenario in NAN_SIMULATION_SCENARIOS_FT:
        scenario_name_for_dir = scenario['name']
        scenario_name_for_file = scenario['name'].replace('%', 'percent')
        print(f"\nProcesando Escenario: {scenario_name_for_dir}")
        sim_data_dir = os.path.join(SIMULATED_DATA_INPUT_ROOT_DIR_FT, city_name_safe, scenario_name_for_dir)
        imputed_scenario_dir = os.path.join(city_imputed_csv_dir, scenario_name_for_dir); os.makedirs(imputed_scenario_dir, exist_ok=True)
        if not os.path.exists(sim_data_dir): continue
        
        for rep in range(1, N_REPETITIONS_FT_EVAL + 1):
            sim_files = glob.glob(os.path.join(sim_data_dir, f"{city_name_safe}_{scenario_name_for_file}_rep{rep}_*_with_nans.csv"))
            if not sim_files: continue
            
            sim_file_path = sim_files[0]
            seed = os.path.basename(sim_file_path).split('_seed')[-1].split('_with_nans.csv')[0]
            df_with_nans = pd.read_csv(sim_file_path, parse_dates=['Date'])
            df_imputed = df_with_nans.copy()
            
            for var in tqdm(ALL_TARGET_COLS_FT, desc=f"  Imputando Rep {rep}", unit="var"):
                model, scaler = finetuned_models_dict.get(var), scalers_dict.get(var)
                if not model or not scaler:
                    if var in df_imputed.columns: df_imputed[var] = df_imputed[var].interpolate().ffill().bfill()
                    continue
                model = model.to('cpu')
                
                univar_with_nans = df_with_nans[[var]].copy()
                nan_mask = univar_with_nans.isnull()
                scaled_zeros = pd.DataFrame(scaler.transform(univar_with_nans.fillna(0)), columns=[var])
                
                imputed_univar = impute_univar_with_finetuned_model(model, scaled_zeros, nan_mask, scaler, SEQ_LEN_MODEL_FT, IMPUTATION_STEP_FT, BATCH_SIZE_IMPUTE_FT, EXPECTED_MODEL_CHANNELS, 'cpu')
                if imputed_univar is not None: df_imputed[var] = imputed_univar[var]

            imputed_filename = os.path.basename(sim_file_path).replace("_with_nans.csv", "_imputed.csv")
            df_imputed.to_csv(os.path.join(imputed_scenario_dir, imputed_filename), index=False)
            
            nan_mask_for_metrics = df_with_nans[ALL_TARGET_COLS_FT].isnull() & ~df_city_ground_truth_ft_reset[ALL_TARGET_COLS_FT].isnull()
            metrics = calculate_metrics_moment(df_city_ground_truth_ft_reset, df_imputed, nan_mask_for_metrics, ALL_TARGET_COLS_FT)
            for var_name, scores in metrics.items():
                all_detailed_metrics_ft_results.append({'City': city, 'Scenario': scenario_name_for_dir, 'Repetition': rep, 'Seed': seed, 'Imputation_Method': MOMENT_METHOD_NAME_FT, 'Variable': var_name, **scores})

            if rep == 1:
                plot_imputation_results_segmented(
                    original_series=df_city_ground_truth_ft_reset['AQI'],
                    imputed_series=df_imputed['AQI'],
                    nan_mask=nan_mask_for_metrics['AQI'],
                    variable_name='AQI', city_name=city, scenario_name=scenario_name_for_dir,
                    output_dir=city_plots_dir
                )

if all_detailed_metrics_ft_results:
    df_metrics = pd.DataFrame(all_detailed_metrics_ft_results)
    sufijo_ft = f"_{sanitize_name(TARGET_CITY_FT)}" if TARGET_CITY_FT else "_ALL_CITIES"
    global_detailed_ft_file = os.path.join(METRICS_FT_OUTPUT_DIR, f'ALL_metrics_detailed_{MOMENT_METHOD_NAME_FT}{sufijo_ft}.csv')
    df_metrics.to_csv(global_detailed_ft_file, index=False)
    print(f"\nMétricas detalladas globales guardadas en: {global_detailed_ft_file}")
    
    print("\nGenerando resúmenes de métricas por variable y ciudad...")
    grouped_by_city_var_ft = df_metrics.groupby(['City', 'Variable'])[['MAE', 'RMSE', 'NRMAE_range', 'NRMSE_range']].mean().reset_index()
    grouped_by_city_var_ft.rename(columns={'MAE': 'Avg_MAE', 'RMSE': 'Avg_RMSE', 'NRMAE_range': 'Avg_NRMAE_range', 'NRMSE_range': 'Avg_NRMSE_range'}, inplace=True)
    for city_name_summary in grouped_by_city_var_ft['City'].unique():
        city_name_safe_summary = sanitize_name(city_name_summary)
        city_metrics_df = grouped_by_city_var_ft[grouped_by_city_var_ft['City'] == city_name_summary]
        variables_summary_path = os.path.join(METRICS_FT_OUTPUT_DIR, f'metrics_variables_summary_{city_name_safe_summary}.csv')
        city_metrics_df.to_csv(variables_summary_path, index=False)
        print(f"  Resumen de variables para {city_name_summary} guardado en: {variables_summary_path}")

    print("\nGenerando resumen combinado de contaminantes...")
    df_contaminants_only_ft = df_metrics[df_metrics['Variable'].isin(CONTAMINANT_COLS_FT)]
    if not df_contaminants_only_ft.empty:
        contaminants_combined_summary_ft = df_contaminants_only_ft.groupby('City')[['MAE', 'RMSE', 'NRMAE_range', 'NRMSE_range']].mean().reset_index()
        contaminants_combined_summary_ft.rename(columns={'MAE': 'Avg_MAE_Conts', 'RMSE': 'Avg_RMSE_Conts', 'NRMAE_range': 'Avg_NRMAE_range_Conts', 'NRMSE_range': 'Avg_NRMSE_range_Conts'}, inplace=True)
        contaminants_summary_path = os.path.join(METRICS_FT_OUTPUT_DIR, 'metrics_contaminants_combined_summary.csv')
        contaminants_combined_summary_ft.to_csv(contaminants_summary_path, index=False)
        print(f"  Resumen contaminantes guardado: {contaminants_summary_path}")

    print("\nGenerando resumen combinado de AQI...")
    df_aqi_only_ft = df_metrics[df_metrics['Variable'] == 'AQI']
    if not df_aqi_only_ft.empty:
        aqi_combined_summary_ft = df_aqi_only_ft.groupby('City')[['MAE', 'RMSE', 'NRMAE_range', 'NRMSE_range']].mean().reset_index()
        aqi_combined_summary_ft.rename(columns={'MAE': 'Avg_MAE_AQI', 'RMSE': 'Avg_RMSE_AQI', 'NRMAE_range': 'Avg_NRMAE_range_AQI', 'NRMSE_range': 'Avg_NRMSE_range_AQI'}, inplace=True)
        aqi_summary_path = os.path.join(METRICS_FT_OUTPUT_DIR, 'metrics_aqi_combined_summary.csv')
        aqi_combined_summary_ft.to_csv(aqi_summary_path, index=False)
        print(f"  Resumen AQI guardado: {aqi_summary_path}")

print(f"\nProceso completado. Tiempo total: {(time.time() - script_start_time_ft) / 60:.2f} minutos.")
if 'log_file' in locals() and log_file and not log_file.closed:
    sys.stdout = original_stdout; sys.stderr = original_stderr;
    log_file.close()
    print(f"Salida de log completada y guardada en: {log_file_path}")

--- INICIO DEL SCRIPT: 2025-06-25 10:02:42 ---

--- 0. Carga de Datos y Modelo Base ---
Datos originales cargados desde: CSVs\Air_Quality.csv
Usando dispositivo: cuda
Cargando modelo MOMENT base: AutonLab/MOMENT-1-large
Modelo base cargado.


==================== PROCESANDO CIUDAD: Brasilia ====================

--- Fine-Tuning para: CO en Brasilia ---

--- Fine-Tuning para: AQI en Brasilia ---

--- Iniciando Fase de Imputación para Brasilia ---

Procesando Escenario: RandomRows_20percent


  Imputando Rep 1: 100%|##########| 2/2 [00:00<00:00, 262.18var/s]


    10 gráficos guardados en: c:\Users\felix\Documents\MÁSTER\TFM\TFM_Ambiental\AIR QUALITY\imputaciones\MOMENT_FT_full_v2_MOMENT_1_large\plots\Brasilia\RandomRows_20percent

Procesando Escenario: BlockRows_12h_20percent


  Imputando Rep 1: 100%|##########| 2/2 [00:00<00:00, 569.57var/s]


    10 gráficos guardados en: c:\Users\felix\Documents\MÁSTER\TFM\TFM_Ambiental\AIR QUALITY\imputaciones\MOMENT_FT_full_v2_MOMENT_1_large\plots\Brasilia\BlockRows_12h_20percent

Procesando Escenario: BlockRows_24h_20percent


...

    10 gráficos guardados en: c:\Users\felix\Documents\MÁSTER\TFM\TFM_Ambiental\AIR QUALITY\imputaciones\MOMENT_FT_full_v2_MOMENT_1_large\plots\Brasilia\BlockRows_36h_20percent

Procesando Escenario: BlockRows_48h_20percent


  Imputando Rep 1: 100%|##########| 2/2 [00:00<00:00, 660.94var/s]


    10 gráficos guardados en: c:\Users\felix\Documents\MÁSTER\TFM\TFM_Ambiental\AIR QUALITY\imputaciones\MOMENT_FT_full_v2_MOMENT_1_large\plots\Brasilia\BlockRows_48h_20percent

Procesando Escenario: BlockRows_60h_20percent


  Imputando Rep 1: 100%|##########| 2/2 [00:00<00:00, 1007.16var/s]


    10 gráficos guardados en: c:\Users\felix\Documents\MÁSTER\TFM\TFM_Ambiental\AIR QUALITY\imputaciones\MOMENT_FT_full_v2_MOMENT_1_large\plots\Brasilia\BlockRows_60h_20percent

Métricas detalladas globales guardadas en: c:\Users\felix\Documents\MÁSTER\TFM\TFM_Ambiental\AIR QUALITY\imputaciones\MOMENT_FT_full_v2_MOMENT_1_large\metricas\ALL_metrics_detailed_MOMENT_FT_full_v2_MOMENT_1_large_Brasilia.csv

Generando resúmenes de métricas por variable y ciudad...
  Resumen de variables para Brasilia guardado en: c:\Users\felix\Documents\MÁSTER\TFM\TFM_Ambiental\AIR QUALITY\imputaciones\MOMENT_FT_full_v2_MOMENT_1_large\metricas\metrics_variables_summary_Brasilia.csv

Generando resumen combinado de contaminantes...
  Resumen contaminantes guardado: c:\Users\felix\Documents\MÁSTER\TFM\TFM_Ambiental\AIR QUALITY\imputaciones\MOMENT_FT_full_v2_MOMENT_1_large\metricas\metrics_contaminants_combined_summary.csv

Generando resumen combinado de AQI...
  Resumen AQI guardado: c:\Users\felix\Documents\M